## Install Required Package (Run Once)

In [1]:
# Install OAuth library - Run this cell once
import subprocess
import sys

try:
    from requests_oauthlib import OAuth2Session
    print("✓ requests-oauthlib already installed")
except ImportError:
    print("Installing requests-oauthlib...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests-oauthlib"])
    print("✓ Installation complete - Please restart the kernel and run cells again")

✓ requests-oauthlib already installed


## Import Libraries

In [2]:
# Required library: requests-oauthlib
# If not installed, run: pip install requests-oauthlib --break-system-packages
import os
os.chdir(r"D:\Scripts\iNaturalist")
import requests
import pandas as pd
import geopandas as gpd
import json
import os
import sys
import time
import webbrowser
from datetime import datetime
from arcgis.gis import GIS

# ============================================================================
# Load Config — all sensitive values live in config.py (never committed to Git)
# ============================================================================
try:
    import config
except ImportError:
    print("ERROR: config.py not found.")
    print("Copy config.example.py to config.py and fill in your credentials.")
    sys.exit(1)

print("✓ Config loaded")

✓ Config loaded


## OAuth Authentication Setup

In [3]:
# ============================================================================
# OAuth Authentication Setup — credentials from config, not hardcoded
# ============================================================================

CLIENT_ID     = config.INAT_CLIENT_ID
CLIENT_SECRET = config.INAT_CLIENT_SECRET
REDIRECT_URI  = 'urn:ietf:wg:oauth:2.0:oob'
TOKEN_FILE    = config.INAT_TOKEN_FILE

AUTHORIZATION_BASE_URL = 'https://www.inaturalist.org/oauth/authorize'
TOKEN_URL              = 'https://www.inaturalist.org/oauth/token'
API_BASE_URL           = 'https://api.inaturalist.org/v1'

def save_token(token):
    with open(TOKEN_FILE, 'w') as f:
        json.dump(token, f)
    print(f"✓ Token saved to {TOKEN_FILE}")

def load_token():
    if not os.path.exists(TOKEN_FILE):
        return None
    try:
        with open(TOKEN_FILE, 'r') as f:
            token = json.load(f)
        if 'expires_at' in token:
            expires_at = datetime.fromtimestamp(token['expires_at'])
            if expires_at < datetime.now():
                print("Saved token has expired, need to re-authenticate")
                os.remove(TOKEN_FILE)
                return None
            hours_remaining = (expires_at - datetime.now()).total_seconds() / 3600
            print(f"✓ Token valid for {hours_remaining:.1f} more hours")
        print("✓ Loaded valid token from file")
        return token
    except Exception as e:
        print(f"Could not load token: {e}")
        return None

def get_authenticated_session():
    token = load_token()
    if token:
        oauth = OAuth2Session(CLIENT_ID, token=token)
        print("✓ Using existing authentication token")
        return oauth

    print("\n" + "="*70)
    print("FIRST-TIME AUTHORIZATION REQUIRED")
    print("="*70)
    print("\nThis is a one-time setup. Token will be saved for future runs.")

    from oauthlib.oauth2 import WebApplicationClient
    client = WebApplicationClient(CLIENT_ID)
    oauth = OAuth2Session(client=client, redirect_uri=REDIRECT_URI)
    authorization_url, state = oauth.authorization_url(AUTHORIZATION_BASE_URL)

    print("\n1. Opening browser for authorization...")
    print(f"   If browser doesn't open, visit:\n   {authorization_url}\n")
    try:
        webbrowser.open(authorization_url)
    except:
        print("   (Could not open browser automatically)")

    print("2. Log into iNaturalist and click 'Authorize'")
    print("3. Copy the authorization code shown and paste it below\n")
    authorization_code = input("Enter the authorization code: ").strip()

    if not authorization_code:
        raise ValueError("No authorization code provided")

    print("\nExchanging authorization code for access token...")
    token_response = requests.post(
        TOKEN_URL,
        data={'grant_type': 'authorization_code', 'code': authorization_code,
              'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET, 'redirect_uri': REDIRECT_URI},
        headers={'Accept': 'application/json'}
    )

    if token_response.status_code != 200:
        raise Exception(f"Token request failed: {token_response.status_code} - {token_response.text}")

    token = token_response.json()
    if 'expires_in' in token:
        from datetime import timedelta
        token['expires_at'] = (datetime.now() + timedelta(seconds=token['expires_in'])).timestamp()

    save_token(token)
    print("✓ Authentication successful!")
    print(f"✓ Token valid for {token.get('expires_in', 7200)/3600:.1f} hours")
    print("✓ Token saved — future runs will use this automatically\n")
    return OAuth2Session(CLIENT_ID, token=token)

try:
    from requests_oauthlib import OAuth2Session
    print("Initializing OAuth authentication...\n")
    session = get_authenticated_session()
    print("="*70)
    print("READY TO MAKE AUTHENTICATED API CALLS")
    print("="*70 + "\n")
except Exception as e:
    print(f"\n✗ Authentication failed: {e}")
    print("⚠ Falling back to unauthenticated requests\n")
    import traceback; traceback.print_exc()
    session = None

Initializing OAuth authentication...

✓ Loaded valid token from file
✓ Using existing authentication token
READY TO MAKE AUTHENTICATED API CALLS



## Check Refresh Token Expiry and Send Warning if Needed

In [4]:
# ============================================================================
# Check Refresh Token Expiry and Send Warning if Needed
# ============================================================================

import sys
import time
sys.path.append(r'D:\Scripts\iNaturalist')
import inat_email_notifications

# Check if token was successfully loaded
if session is not None and hasattr(session, 'token') and session.token:
    token = session.token
    
    # Check if refresh token is about to expire
    if 'refresh_token' in token and 'expires_at' in token:
        # Refresh tokens typically last ~180 days (6 months)
        # Calculate days remaining (rough estimate based on access token age)
        refresh_token_expires_at = token.get('created_at', time.time()) + (180 * 24 * 3600)
        days_remaining = (refresh_token_expires_at - time.time()) / (24 * 3600)
        
        if days_remaining < inat_email_notifications.REFRESH_TOKEN_WARNING_DAYS:
            print(f"⚠️ Refresh token expires in {int(days_remaining)} days - sending warning email...")
            inat_email_notifications.send_refresh_token_warning(
                int(days_remaining),
                inat_email_notifications.ADMIN_EMAIL
            )
    else:
        print("✓ Token information not available for expiry check")
else:
    print("⚠️ No authenticated session available - skipping refresh token expiry check")

✓ Token information not available for expiry check


## Check Token Status

In [5]:
# ============================================================================
# Check Token Status 
# ============================================================================

import os
import json
from datetime import datetime

# Token file path (same as defined in OAuth setup)
TOKEN_FILE = 'inat_token.json'

if os.path.exists(TOKEN_FILE):
    try:
        with open(TOKEN_FILE, 'r') as f:
            token = json.load(f)
        
        print("=" * 70)
        print("TOKEN STATUS")
        print("=" * 70)
        
        if 'expires_at' in token:
            expires_at = datetime.fromtimestamp(token['expires_at'])
            now = datetime.now()
            
            if expires_at > now:
                time_remaining = expires_at - now
                hours_remaining = time_remaining.total_seconds() / 3600
                minutes_remaining = (time_remaining.total_seconds() % 3600) / 60
                
                print(f"✓ Token is valid")
                print(f"  Time remaining: {int(hours_remaining)}h {int(minutes_remaining)}m")
                print(f"  Expires at: {expires_at.strftime('%Y-%m-%d %H:%M:%S')}")
                
                # Show token details
                if 'access_token' in token:
                    print(f"  Access token: {token['access_token'][:20]}...{token['access_token'][-10:]}")
                if 'scope' in token:
                    print(f"  Scopes: {token['scope']}")
                
            else:
                print("✗ Token has expired")
                print(f"  Expired at: {expires_at.strftime('%Y-%m-%d %H:%M:%S')}")
                print("  Run the OAuth Authentication Setup cell to get a new token")
        else:
            print("Token file exists but missing expiration info")
            print("Token may still be valid, but expiration cannot be determined")
        
        print("=" * 70)
        
    except json.JSONDecodeError:
        print("✗ Error: Token file is corrupted")
        print(f"  Delete {TOKEN_FILE} and re-authenticate")
    except Exception as e:
        print(f"✗ Error reading token: {e}")
else:
    print("=" * 70)
    print("TOKEN STATUS")
    print("=" * 70)
    print("No token file found")
    print(f"  Expected location: {os.path.abspath(TOKEN_FILE)}")
    print("  Run the OAuth Authentication Setup cell to authenticate")
    print("=" * 70)

TOKEN STATUS
Token file exists but missing expiration info
Token may still be valid, but expiration cannot be determined


## API Query

In [6]:
# Pull a single page to verify API connectivity and auth
project_id = config.INAT_PROJECT_ID

url = f"{API_BASE_URL}/observations"
params = {
    "project_id": project_id,
    "per_page": 5,
    "page": 1,
    "order": "desc",
    "order_by": "created_at",
}

response = session.get(url, params=params) if session else requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print(f"✓ API working — total observations in project: {data.get('total_results', 'unknown')}")
    if data.get('results'):
        print(f"  Sample: {data['results'][0].get('species_guess','?')} on {data['results'][0].get('observed_on','?')}")
else:
    print(f"✗ API request failed: {response.status_code}")

✓ API working — total observations in project: 517
  Sample: Woolly nightshade on 2026-03-16


## Define Programme Species Lists

In [7]:
# Define programme species lists with taxon IDs
programme_species = {
    'Progressive Containment': {
        'names': [
            'Banana Passionfruit', 'Boneseed', "Darwin's Barberry", 
            'Evergreen Buckthorn', 'Grey Willow', 'Moth Plant', 
            "Old Man's Beard", 'Lodgepole Pine', 'Mountain Pine', 
            'Scots Pine', 'Dwarf Mountain Pine',
            # Add scientific names and variants
            'Passiflora tarminiana', 'Passiflora tripartita', 
            'Passiflora tarminiana × tripartita', 'Passiflora tripartita mollissima',
            'Clematis vitalba', 'Osteospermum moniliferum', 
            'Osteospermum moniliferum moniliferum',
            'Elkea'  # Taxonomic section for Banana Passionfruit group
        ],
        'taxon_ids': [1442036, 61400, 75751, 82856, 168326, 75499, 67747, 
                      48934, 135715, 58722, 135727,  
                      # Add taxon IDs for variants
                      1442036, 1442037, 1442036, 404420, 412975, 133171, 133168, # Passiflora variants
                      160697, 61400, 61400, 600447,  # Clematis vitalba, Osteospermum variants
                      1442036]  # Elkea section (use Banana Passionfruit taxon_id as proxy)
    },
    'Eradication': {
        'names': [
            'Alligator Weed', 'Blue Passionflower', 'Cathedral Bells',
            'Chilean Rhubarb', 'Giant Rhubarb', 'Chinese Pennisetum',
            'Climbing Alstroemeria', 'Climbing Spindleberry', 'Himalayan Balsam',
            'Chilean Needle Grass', 'Mexican Feathergrass', 'Serrated Tussock',
            'Purple Loosestrife', 'Queensland Poplar', 'Black Cherry', 
            'Rum Cherry', 'Senegal Tea', 'Saltmarsh Cordgrass',
            'Sporobolus alterniflorus × foliosus', 'Common Cordgrass',
            'Small Cord-Grass', 'Dense-flowered Cord Grass', 
            "Townsend's Cord-Grass", 'Woolly Nightshade',
            # Add scientific names
            'Cobaea scandens', 'Reynoutria japonica', 'Bomarea multiflora',
            'Japanese Knotweed'
        ],
        'taxon_ids': [75386, 51454, 164333, 77310, 1432862, 1030051, 283354, 
                      64540, 47892, 165658, 165660, 165660, 61321, 369932, 
                      54834, 54834, 407490, 772903, 773011, 772983, 772998, 
                      772990, 773009, 133287,
                      # Add taxon IDs
                      164333, 430005, 430005]  # Cobaea, Reynoutria, Bomarea
    },
    'Exclusion': {
        'names': [
            'African Feathergrass', 'California Bulrush', 'Chilean Needle Grass',
            'Heath Rush', 'Humped Bladderwort', 'Manchurian Wild Rice',
            'Noogoora Burr', 'Common Reed', 'Saffron Thistle', 'Arrowhead',
            'Sweet Pittosporum', 'Tussock Hawkweed',
            # Add scientific names
            'Sagittaria platyphylla'
        ],
        'taxon_ids': [1080223, 47159, 165658, 164948, 79464, 407086, 57920, 
                      64237, 64232, 48071, 51594, 409445, 69816]  
    }
}

print("Programme species lists defined with taxon IDs")
print(f"Progressive Containment: {len(programme_species['Progressive Containment']['taxon_ids'])} species/variants")
print(f"Eradication: {len(programme_species['Eradication']['taxon_ids'])} species/variants")
print(f"Exclusion: {len(programme_species['Exclusion']['taxon_ids'])} species/variants")

Programme species lists defined with taxon IDs
Progressive Containment: 23 species/variants
Eradication: 27 species/variants
Exclusion: 13 species/variants


## Pull All Observations

In [8]:
project_id = 266532
url = f"{API_BASE_URL}/observations"

# Use authenticated session if available, otherwise fall back to unauthenticated
if session is None:
    print("WARNING: Using unauthenticated requests (no access to obscured coordinates)")
    http_client = requests
else:
    print("Using authenticated session (access to trusted coordinates)")
    http_client = session

params = {
    "project_id": project_id,
    "per_page": 200,
    "page": 1,
    "order": "desc",
    "order_by": "created_at",
    #"verifiable": "true"
}

observations = []
while True:
    response = http_client.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Error: HTTP {response.status_code}")
        print(response.text[:500])
        break
        
    data = response.json()
    if 'results' not in data or not data['results']:
        break
        
    observations.extend(data['results'])
    print(f"Fetched page {params['page']} — total observations so far: {len(observations)}")
    
    # Check if we've reached the last page
    if len(data['results']) < params['per_page']:
        break
        
    params['page'] += 1

print(f"\nTotal observations fetched: {len(observations)}")

Using authenticated session (access to trusted coordinates)
Fetched page 1 — total observations so far: 200
Fetched page 2 — total observations so far: 400
Fetched page 3 — total observations so far: 517

Total observations fetched: 517


## Create Lookup Dictionaries

In [9]:
# Create lookup dictionaries for taxon_id -> programme and name -> programme
taxon_to_programme = {}
name_to_programme = {}

for programme, species_data in programme_species.items():
    for taxon_id in species_data['taxon_ids']:
        taxon_to_programme[taxon_id] = programme
    for name in species_data['names']:
        # Normalize names for matching (lowercase, strip)
        name_to_programme[name.lower().strip()] = programme

print(f"Created lookup for {len(taxon_to_programme)} taxon IDs")
print(f"Created lookup for {len(name_to_programme)} species names")

Created lookup for 53 taxon IDs
Created lookup for 59 species names


## Build the DataFrame with Programme Categorization

In [10]:

rows = []
for obs in observations:
    # Get coordinates with authentication priority:
    # 1. Private coordinates (available via authentication for obscured observations)
    # 2. Public coordinates (fallback)
    
    private_lat = obs.get('private_latitude')
    private_lon = obs.get('private_longitude')
    
    # Check geojson for public/obscured coordinates
    geojson_coords = obs.get('geojson', {}).get('coordinates', [None, None])
    public_lon, public_lat = geojson_coords[0], geojson_coords[1]
    
    # Use private if available (better accuracy for obscured obs), otherwise public
    lat = private_lat if private_lat is not None else public_lat
    lon = private_lon if private_lon is not None else public_lon
    
    # Track whether coordinates are obscured and if we have private access
    is_obscured = obs.get('obscured', False)
    geoprivacy = obs.get('geoprivacy')
    has_private_coords = private_lat is not None and private_lon is not None
    
    # Get photos (up to 3) and convert to original size
    photos = obs.get('photos', [])
    photo1 = photos[0]['url'].replace('square', 'original') if len(photos) >= 1 else None
    photo2 = photos[1]['url'].replace('square', 'original') if len(photos) >= 2 else None
    photo3 = photos[2]['url'].replace('square', 'original') if len(photos) >= 3 else None
    
    # Process observation fields
    of_list = obs.get('observation_fields') or []
    of_dict = {}
    for f in of_list:
        key = f.get('name') or f.get('field_id') or f.get('observation_field_id')
        of_dict[key] = f.get('value')
    
    # Get taxon info
    taxon = obs.get('taxon', {})
    taxon_id = taxon.get('id')
    taxon_name = taxon.get('name', '')
    species_guess = obs.get('species_guess', '')
    
    # Determine programme category
    programme = None
    
    # First try matching by taxon_id (most reliable)
    if taxon_id and taxon_id in taxon_to_programme:
        programme = taxon_to_programme[taxon_id]
    # Then try matching by taxon name
    elif taxon_name and taxon_name.lower().strip() in name_to_programme:
        programme = name_to_programme[taxon_name.lower().strip()]
    # Finally try species_guess
    elif species_guess and species_guess.lower().strip() in name_to_programme:
        programme = name_to_programme[species_guess.lower().strip()]

    # Get annotations and extract Flowers/Fruits
    annotations = obs.get('annotations', [])
    flowers_fruits = None
    for annotation in annotations:
        # Annotation IDs: 12 = "Plant Phenology", value 13 = "Flowering", 14 = "Fruiting"
        if annotation.get('controlled_attribute_id') == 12:  # Plant Phenology
            value_id = annotation.get('controlled_value_id')
            if value_id == 13:
                flowers_fruits = 'Flowering'
            elif value_id == 14:
                flowers_fruits = 'Fruiting'
            elif value_id == 15:
                flowers_fruits = 'Flower Budding'
    
    rows.append({
        "id": obs.get("id"),
        "observation_url": f"https://www.inaturalist.org/observations/{obs.get('id')}",
        "taxon_id": taxon_id,
        "taxon_name": taxon_name,
        "species_guess": species_guess,
        "programme": programme,
        "observed_on": obs.get("observed_on"),
        "description": obs.get('description'),
        "user_login": obs.get("user", {}).get("login"),
        "user_name": obs.get("user", {}).get("name"),
        "place_guess": obs.get('place_guess'),
        "quality_grade": obs.get("quality_grade"),
        "geoprivacy": geoprivacy,
        "is_obscured": is_obscured,
        "has_private_coords": has_private_coords,
        "latitude": lat,
        "longitude": lon,
        "photoURL_1": photo1,
        "photoURL_2": photo2,
        "photoURL_3": photo3,
        "private_place_guess": obs.get('private_place_guess'),
        "private_latitude": obs.get('private_latitude'),
        "private_longitude": obs.get('private_longitude'),
        "obs_fields": of_dict,
        "flowers_fruits": flowers_fruits 
    })

df = pd.DataFrame(rows)
def categorize_observation_date(date_string):
    """
    Categorize observation date into user-friendly time ranges
    for easy filtering in Field Maps
    """
    if pd.isna(date_string) or date_string == '':
        return 'Unknown Date'
    
    try:
        # Parse the observation date
        obs_date = datetime.strptime(date_string, '%Y-%m-%d').date()
        today = datetime.now().date()
        days_ago = (today - obs_date).days
        
        # Categorize by age
        if days_ago < 0:
            return 'Future Date'  # Data error
        elif days_ago == 0:
            return 'Today'
        elif days_ago <= 7:
            return 'Last 7 Days'
        elif days_ago <= 30:
            return 'Last 30 Days'
        elif days_ago <= 90:
            return 'Last 3 Months'
        elif days_ago <= 365:
            return 'Last Year'
        else:
            return 'Older than 1 Year'
    except:
        return 'Invalid Date'

# Apply the categorization
df['date_category'] = df['observed_on'].apply(categorize_observation_date)

print(f"\nDate categories added:")
print(df['date_category'].value_counts().sort_index())

df.head()


Date categories added:
date_category
Last 3 Months         20
Last 30 Days           9
Last 7 Days            3
Last Year             64
Older than 1 Year    416
Unknown Date           5
Name: count, dtype: int64


,id,observation_url,taxon_id,taxon_name,species_guess,programme,observed_on,description,user_login,user_name,...,longitude,photoURL_1,photoURL_2,photoURL_3,private_place_guess,private_latitude,private_longitude,obs_fields,flowers_fruits,date_category
0,343225422,https://www.inaturalist.org/observations/34322...,133287,Solanum mauritianum,Woolly nightshade,Eradication,2026-03-16,None,mmorgan_nz,Michael Morgan,...,175.610222,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,{},None,Last 7 Days
1,343067581,https://www.inaturalist.org/observations/34306...,75499,Araujia sericifera,Moth Vine,Progressive Containment,2026-03-12,"Not common in this city, but if these pods rip...",cco,Colin Ogle,...,175.049684,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,{},None,Last 30 Days
2,343040301,https://www.inaturalist.org/observations/34304...,77310,Gunnera tinctoria,Chilean rhubarb,Eradication,2026-03-15,None,mmorgan_nz,Michael Morgan,...,175.706704,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,{},None,Last 7 Days
3,342669254,https://www.inaturalist.org/observations/34266...,160697,Clematis vitalba,Old man's beard,Progressive Containment,2026-03-13,,mefisher,,...,175.781495,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,None,{},None,Last 7 Days
4,342153773,https://www.inaturalist.org/observations/34215...,160697,Clematis vitalba,Old man's beard,Progressive Containment,2026-03-10,None,heemi_tuutiri,Hēmi Tūtiri,...,175.100726,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,None,{},None,Last 30 Days


## Verify Authentication Effectiveness

In [11]:

print("=" * 70)
print("COORDINATE ACCESS STATISTICS")
print("=" * 70) 

print(f"\nTotal observations: {len(df)}")
print(f"Obscured observations: {df['is_obscured'].sum()}")
print(f"Observations with geoprivacy set: {df['geoprivacy'].notna().sum()}")
print(f"Observations with private coordinates: {df['has_private_coords'].sum()}")

print("\n--- Authentication Effectiveness ---")
obscured_count = df['is_obscured'].sum()
obscured_with_private = df[df['is_obscured'] & df['has_private_coords']].shape[0]

if obscured_count > 0:
    success_rate = (obscured_with_private / obscured_count) * 100
    print(f"Obscured observations with private coordinates: {obscured_with_private}/{obscured_count}")
    print(f"Success rate: {success_rate:.1f}%")
    
    if success_rate == 100:
        print("✓ EXCELLENT: All obscured observations have private coordinates!")
    elif success_rate > 0:
        print("✓ GOOD: Authentication is working for some obscured observations")
        print("  Note: Some observations may be obscured by other users/projects")
    else:
        print("WARNING: No private coordinates obtained for obscured observations")
        print("  Check that your app has been granted trust by the project")
else:
    print("ℹ No obscured observations in dataset yet")
    print("  Authentication is configured and ready for when they appear")

# Show examples of obscured observations (if any)
if obscured_count > 0:
    print("\n--- Sample Obscured Observations ---")
    obscured_sample = df[df['is_obscured']][['id', 'taxon_name', 'geoprivacy', 'is_obscured', 'has_private_coords']].head(5)
    print(obscured_sample.to_string(index=False))

print("\n" + "=" * 70)

COORDINATE ACCESS STATISTICS

Total observations: 517
Obscured observations: 8
Observations with geoprivacy set: 8
Observations with private coordinates: 0

--- Authentication Effectiveness ---
Obscured observations with private coordinates: 0/8
Success rate: 0.0%
  Check that your app has been granted trust by the project

--- Sample Obscured Observations ---
       id          taxon_name geoprivacy  is_obscured  has_private_coords
282997977   Lythrum salicaria   obscured         True               False
258292724               Elkea   obscured         True               False
246085964 Solanum mauritianum   obscured         True               False
155063546    Clematis vitalba   obscured         True               False
134827757               Elkea   obscured         True               False



## Summary Statistics

In [12]:
# Summary statistics
print("=== Observations by Programme ===")
print(df['programme'].value_counts(dropna=False))
print(f"\nTotal observations: {len(df)}")
print(f"Categorized observations: {df['programme'].notna().sum()}")
print(f"Uncategorized observations: {df['programme'].isna().sum()}")

# Show uncategorized species
if df['programme'].isna().any():
    print("\n=== Uncategorized Species ===")
    uncategorized = df[df['programme'].isna()][['taxon_name', 'species_guess']].drop_duplicates()
    print(uncategorized)

# Show coordinate access summary
print("\n=== Coordinate Access Summary ===")
print(f"Total observations: {len(df)}")
print(f"With private coordinates: {df['has_private_coords'].sum()} ({df['has_private_coords'].sum()/len(df)*100:.1f}%)")
print(f"Obscured observations: {df['is_obscured'].sum()}")

=== Observations by Programme ===
programme
Progressive Containment    377
Eradication                137
Exclusion                    3
Name: count, dtype: int64

Total observations: 517
Categorized observations: 517
Uncategorized observations: 0

=== Coordinate Access Summary ===
Total observations: 517
With private coordinates: 0 (0.0%)
Obscured observations: 8


## Convert to a GeoDataFrame

In [13]:
# Convert to GeoDataFrame and reproject to NZTM2000 (EPSG:2193)
from shapely.geometry import Point
import geopandas as gpd

# 1) Drop rows missing coords (latitude/longitude)
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()

# 2) Create geometry in WGS84 (EPSG:4326)
geometry = [Point(xy) for xy in zip(df_clean['longitude'], df_clean['latitude'])]
gdf = gpd.GeoDataFrame(df_clean, geometry=geometry, crs="EPSG:4326")

# 3) Reproject to NZTM2000 (EPSG:2193)
gdf_nztm = gdf.to_crs(epsg=2193)

# 4) Quick sanity checks
print("Original count:", len(df_clean))
print("GeoDataFrame count:", len(gdf_nztm))
print("CRS after reprojection:", gdf_nztm.crs)
print("\nProgramme distribution in GeoDataFrame:")
print(gdf_nztm['programme'].value_counts(dropna=False))

# show first few rows
gdf_nztm.head()

Original count: 517
GeoDataFrame count: 517
CRS after reprojection: EPSG:2193

Programme distribution in GeoDataFrame:
programme
Progressive Containment    377
Eradication                137
Exclusion                    3
Name: count, dtype: int64


,id,observation_url,taxon_id,taxon_name,species_guess,programme,observed_on,description,user_login,user_name,...,photoURL_1,photoURL_2,photoURL_3,private_place_guess,private_latitude,private_longitude,obs_fields,flowers_fruits,date_category,geometry
0,343225422,https://www.inaturalist.org/observations/34322...,133287,Solanum mauritianum,Woolly nightshade,Eradication,2026-03-16,None,mmorgan_nz,Michael Morgan,...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,{},None,Last 7 Days,POINT (1821698.549 5530801.860)
1,343067581,https://www.inaturalist.org/observations/34306...,75499,Araujia sericifera,Moth Vine,Progressive Containment,2026-03-12,"Not common in this city, but if these pods rip...",cco,Colin Ogle,...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,{},None,Last 30 Days,POINT (1775146.030 5578020.609)
2,343040301,https://www.inaturalist.org/observations/34304...,77310,Gunnera tinctoria,Chilean rhubarb,Eradication,2026-03-15,None,mmorgan_nz,Michael Morgan,...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,{},None,Last 7 Days,POINT (1829715.692 5524724.761)
3,342669254,https://www.inaturalist.org/observations/34266...,160697,Clematis vitalba,Old man's beard,Progressive Containment,2026-03-13,,mefisher,,...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,None,{},None,Last 7 Days,POINT (1838039.987 5587656.057)
4,342153773,https://www.inaturalist.org/observations/34215...,160697,Clematis vitalba,Old man's beard,Progressive Containment,2026-03-10,None,heemi_tuutiri,Hēmi Tūtiri,...,https://inaturalist-open-data.s3.amazonaws.com...,None,None,None,None,None,{},None,Last 30 Days,POINT (1779546.441 5579555.911)


## Map Species to SMU Layers

In [14]:
# Mapping of taxon_name (scientific name) to SMU feature class names
taxon_name_to_smu = {
    # Progressive Containment - Banana Passionfruit group
    'Passiflora tripartita': 'SMU_Banana_Passionfruit',
    'Passiflora tarminiana': 'SMU_Banana_Passionfruit',
    'Passiflora tripartita mollissima': 'SMU_Banana_Passionfruit',
    'Passiflora tripartita azuayensis': 'SMU_Banana_Passionfruit',
    'Elkea': 'SMU_Banana_Passionfruit',  # Taxonomic section
    'Tacsonia': 'SMU_Banana_Passionfruit',  # Old genus name
    'Osteospermum moniliferum': 'SMU_Boneseed',
    'Osteospermum moniliferum moniliferum': 'SMU_Boneseed',
    'Berberis darwinii': 'SMU_Darwins_barberry',
    'Rhamnus alaternus': None,  # Evergreen Buckthorn - NO SMU LAYER
    'Salix cinerea': None,  # Grey Willow - NO SMU LAYER
    'Araujia hortorum': 'SMU_Mothplant',
    'Araujia sericifera': 'SMU_Mothplant',
    'Clematis vitalba': 'SMU_Old_mans_beard',
    'Pinus contorta': 'SMU_Pest_conifers',
    'Pinus uncinata': 'SMU_Pest_conifers',
    'Pinus mugo': 'SMU_Pest_conifers',
    'Pinus sylvestris': 'SMU_Pest_conifers',
    
    # Eradication
    'Alternanthera philoxeroides': 'SMU_Master_Lay_Alligator_Weed',
    'Passiflora caerulea': 'SMU_Blue_passion_flower',
    'Cobaea scandens': 'SMU_Cathedral_Bells',
    'Gunnera tinctoria': 'SMU_Gunnera',
    'Gunnera manicata': 'SMU_Gunnera',
    'Pennisetum alopecuroides': 'SMU_Chinese_pennisetum',
    'Bomarea multiflora': 'SMU_Climbing_Alstromeria',
    'Celastrus orbiculatus': 'SMU_Climbing_Spindleberry',
    'Impatiens glandulifera': 'SMU_Himalayan_Balsam',
    'Nasella neesiana': 'SMU_Nasella_Tussock',
    'Nassella trichotoma': 'SMU_Nasella_Tussock',
    'Nassella tenuissima': 'SMU_Nasella_Tussock',
    'Lythrum salicaria': 'SMU_Purple_loosestrife',
    'Homalanthus populifolius': 'SMU_Queensland_poplar',
    'Prunus serotina': 'SMU_Rum_Cherry',
    'Gymnocoronis spilanthoides': 'SMU_Senegal_tea',
    'Spartina alterniflora': 'SMU_Spartina',
    'Spartina anglica': 'SMU_Spartina',
    'Sporobolus anglicus': 'SMU_Spartina',
    'Sporobolus × townsendii': 'SMU_Spartina',
    'Spartina densiflora': 'SMU_Spartina',
    'Spartina maritima': 'SMU_Spartina',
    'Spartina patens': 'SMU_Spartina',
    'Solanum mauritianum': 'SMU_Woolly_Nightshade',
    'Reynoutria japonica': 'SMU_knotweed',
    'Fallopia japonica': 'SMU_knotweed',
    
    # Exclusion
    'Pennisetum macrourum': 'SMU_African_Feather_Grass',
    'Sagittaria platyphylla': 'SMU_Sagittaria',
    'Sagittaria montevidensis': 'SMU_Arrowhead',
    'Bolboschoenus robustus': None,
    'Juncus squarrosus': None,
    'Utricularia gibba': None,
    'Zizania latifolia': None,
    'Xanthium occidentale': None,
    'Phragmites australis': None,
    'Carthamus lanatus': None,
    'Pittosporum undulatum': None,
    'Hieracium lepidulum': None,
}

print(f"Mapped {len([v for v in taxon_name_to_smu.values() if v is not None])} taxon names to SMU layers")
print(f"({len([v for v in taxon_name_to_smu.values() if v is None])} species have no SMU layer)")

Mapped 45 taxon names to SMU layers
(11 species have no SMU layer)


## Operating Area to Operator Mapping

In [15]:
# Mapping of operatingArea to Name (operator) — loaded from config
# Staff names are kept in config.py 
operating_area_to_operator = config.AREA_TO_STAFF

print(f"Operating area mapping loaded from config: {len(operating_area_to_operator)} areas")

Operating area mapping created for 7 areas


## Split GeoDataFrame by Programme Type

In [16]:
# Split the GeoDataFrame into three separate GeoDataFrames by programme
gdf_progressive = gdf_nztm[gdf_nztm['programme'] == 'Progressive Containment'].copy()
gdf_eradication = gdf_nztm[gdf_nztm['programme'] == 'Eradication'].copy()
gdf_exclusion = gdf_nztm[gdf_nztm['programme'] == 'Exclusion'].copy()

print("Progressive Containment observations:", len(gdf_progressive))
print("Eradication observations:", len(gdf_eradication))
print("Exclusion observations:", len(gdf_exclusion))

Progressive Containment observations: 377
Eradication observations: 137
Exclusion observations: 3


## Add SMU Layer Name to GeoDataFrame

In [17]:
# Add a column to each GeoDataFrame indicating which SMU layer to use
def assign_smu_layer(row):
    """Assign SMU layer name based on taxon_name (scientific name)"""
    # Match based on standardized taxon_name
    if pd.notna(row['taxon_name']) and row['taxon_name'] in taxon_name_to_smu:
        return taxon_name_to_smu[row['taxon_name']]
    # No SMU layer available
    return None

gdf_progressive['smu_layer'] = gdf_progressive.apply(assign_smu_layer, axis=1)
gdf_eradication['smu_layer'] = gdf_eradication.apply(assign_smu_layer, axis=1)
gdf_exclusion['smu_layer'] = gdf_exclusion.apply(assign_smu_layer, axis=1)

# Check how many observations have SMU layers available
print("Progressive Containment:")
print(f"  With SMU layer: {gdf_progressive['smu_layer'].notna().sum()}")
print(f"  Without SMU layer: {gdf_progressive['smu_layer'].isna().sum()}")

print("\nEradication:")
print(f"  With SMU layer: {gdf_eradication['smu_layer'].notna().sum()}")
print(f"  Without SMU layer: {gdf_eradication['smu_layer'].isna().sum()}")

print("\nExclusion:")
print(f"  With SMU layer: {gdf_exclusion['smu_layer'].notna().sum()}")
print(f"  Without SMU layer: {gdf_exclusion['smu_layer'].isna().sum()}")

Progressive Containment:
  With SMU layer: 338
  Without SMU layer: 39

Eradication:
  With SMU layer: 137
  Without SMU layer: 0

Exclusion:
  With SMU layer: 1
  Without SMU layer: 2


## Add Normalized Species Name Field

In [18]:
# Create a mapping of taxon_name to standardized common names for display
taxon_name_to_common = {
    # Progressive Containment - Banana Passionfruit group
    'Passiflora tripartita': 'Banana Passionfruit',
    'Passiflora tarminiana': 'Banana Passionfruit',
    'Passiflora tripartita mollissima': 'Banana Passionfruit',
    'Passiflora tripartita azuayensis': 'Banana Passionfruit',
    'Elkea': 'Banana Passionfruit',  # Taxonomic section
    'Tacsonia': 'Banana Passionfruit',  # Old genus name
    'Osteospermum moniliferum': 'Boneseed',
    'Osteospermum moniliferum moniliferum': 'Boneseed',
    'Berberis darwinii': "Darwin's Barberry",
    'Rhamnus alaternus': 'Evergreen Buckthorn',
    'Salix cinerea': 'Grey Willow',
    'Araujia hortorum': 'Moth Plant',
    'Araujia sericifera': 'Moth Plant',
    'Clematis vitalba': "Old Man's Beard",
    'Pinus contorta': 'Pest Conifers',
    'Pinus uncinata': 'Pest Conifers',
    'Pinus mugo': 'Pest Conifers',
    'Pinus sylvestris': 'Pest Conifers',
    
    # Eradication
    'Alternanthera philoxeroides': 'Alligator Weed',
    'Passiflora caerulea': 'Blue Passionflower',
    'Cobaea scandens': 'Cathedral Bells',
    'Gunnera tinctoria': 'Chilean Rhubarb',
    'Gunnera manicata': 'Chilean Rhubarb',
    'Pennisetum alopecuroides': 'Chinese Pennisetum',
    'Bomarea multiflora': 'Climbing Alstroemeria',
    'Celastrus orbiculatus': 'Climbing Spindleberry',
    'Impatiens glandulifera': 'Himalayan Balsam',
    'Nasella neesiana': 'Nassella Tussock',
    'Nassella trichotoma': 'Nassella Tussock',
    'Nassella tenuissima': 'Nassella Tussock',
    'Lythrum salicaria': 'Purple Loosestrife',
    'Homalanthus populifolius': 'Queensland Poplar',
    'Prunus serotina': 'Rum Cherry',
    'Gymnocoronis spilanthoides': 'Senegal Tea',
    'Spartina alterniflora': 'Spartina',
    'Spartina anglica': 'Spartina',
    'Sporobolus anglicus': 'Spartina',
    'Sporobolus × townsendii': 'Spartina',
    'Spartina densiflora': 'Spartina',
    'Spartina maritima': 'Spartina',
    'Spartina patens': 'Spartina',
    'Solanum mauritianum': 'Woolly Nightshade',
    'Reynoutria japonica': 'Japanese Knotweed',
    'Fallopia japonica': 'Japanese Knotweed',
    
    # Exclusion
    'Pennisetum macrourum': 'African Feathergrass',
    'Sagittaria platyphylla': 'Arrowhead',
    'Sagittaria montevidensis': 'Arrowhead',
    'Bolboschoenus robustus': 'California Bulrush',
    'Juncus squarrosus': 'Heath Rush',
    'Utricularia gibba': 'Humped Bladderwort',
    'Zizania latifolia': 'Manchurian Wild Rice',
    'Xanthium occidentale': 'Noogoora Burr',
    'Phragmites australis': 'Common Reed',
    'Carthamus lanatus': 'Saffron Thistle',
    'Pittosporum undulatum': 'Sweet Pittosporum',
    'Hieracium lepidulum': 'Tussock Hawkweed',
}

# Add standardized common name field to each GeoDataFrame
def get_standardized_name(row):
    """Get standardized common name based on taxon_name"""
    if pd.notna(row['taxon_name']) and row['taxon_name'] in taxon_name_to_common:
        return taxon_name_to_common[row['taxon_name']]
    # Fall back to species_guess if no standardized name available
    return row['species_guess'] if pd.notna(row['species_guess']) else row['taxon_name']

gdf_progressive['speciesName'] = gdf_progressive.apply(get_standardized_name, axis=1)
gdf_eradication['speciesName'] = gdf_eradication.apply(get_standardized_name, axis=1)
gdf_exclusion['speciesName'] = gdf_exclusion.apply(get_standardized_name, axis=1)

print("Added standardized speciesName field")
print(f"Total species mapped: {len(taxon_name_to_common)}")

Added standardized speciesName field
Total species mapped: 56


## Load Spatial Join SMU Layers And Update Main GeoDataFrame

In [19]:
import os
import geopandas as gpd
import pandas as pd

# ArcPy & shapely for SDE -> GeoDataFrame conversion
import arcpy
import shapely.wkb

# GDB / SDE paths
# Paths from config — not hardcoded
gdb_path = config.GDB_SMU_PATH
sde_path = config.SDE_PATH

def arc_sde_feature_to_gdf(sde_conn_path, feature_path_within_sde):
    """
    Read a feature class from an .sde connection using arcpy and return a GeoDataFrame.
    - sde_conn_path: path to the .sde file (UNC or local).
    - feature_path_within_sde: the rest of the path inside the SDE, e.g.
      'biosecurity.GISADMIN.Weeds_ProgressiveContainmentMappedZone' OR
      r"\\...sde\schema\FeatureClass"
      If the input already contains the .sde at the front, this function will attempt to join them.
    Returns: geopandas.GeoDataFrame
    """
    # build full path if needed
    if sde_conn_path in feature_path_within_sde:
        full_path = feature_path_within_sde
    else:
        full_path = fr"{sde_conn_path}\{feature_path_within_sde}"
    try:
        desc = arcpy.Describe(full_path)
    except Exception as e:
        raise RuntimeError(f"arcpy.Describe failed for '{full_path}': {e}")

    sr = desc.spatialReference
    epsg = None
    try:
        wkid = getattr(sr, "factoryCode", None)
        if wkid and int(wkid) != 0:
            epsg = int(wkid)
    except Exception:
        epsg = None

    # list fields excluding geometry and OID types
    fields = [f.name for f in arcpy.ListFields(full_path) if f.type not in ('Geometry', 'OID')]
    cursor_fields = ['SHAPE@WKB'] + fields

    features = []
    with arcpy.da.SearchCursor(full_path, cursor_fields) as cursor:
        for row in cursor:
            geom_wkb = row[0]
            # arcpy returns bytes for SHAPE@WKB; convert to shapely geometry
            geom = None
            if geom_wkb:
                try:
                    geom = shapely.wkb.loads(geom_wkb)
                except Exception:
                    # try converting memoryview -> bytes if necessary
                    geom = shapely.wkb.loads(bytes(geom_wkb))
            attr = {fld: val for fld, val in zip(fields, row[1:])}
            attr['geometry'] = geom
            features.append(attr)

    gdf = gpd.GeoDataFrame(features, crs=(f"EPSG:{epsg}" if epsg else None))
    return gdf

def spatial_join_with_smu(gdf, programme_name, distance_threshold=100):
    """
    Perform spatial join for observations with their corresponding SMU layers
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Input observations
    programme_name : str
        Name of the programme ('Progressive Containment', 'Eradication', 'Exclusion')
    distance_threshold : float
        Maximum distance in meters for nearest join fallback (default: 100m)
        
    For Progressive Containment without SMU (Evergreen Buckthorn & Grey Willow):
        - Use Weeds_ProgressiveContainmentMappedZone for Designation
        - Use OperatingAreas for operatingArea and Name
    For observations still without match: use OperatingAreas as final fallback
    For observations near but outside boundaries: use distance-based nearest join
    """
    # Initialize columns for SMU attributes if they don't exist
    if 'operatingArea' not in gdf.columns:
        gdf['operatingArea'] = None
    if 'Name' not in gdf.columns:
        gdf['Name'] = None
    if 'SMU_Name' not in gdf.columns:
        gdf['SMU_Name'] = None
    if 'Designation' not in gdf.columns:
        gdf['Designation'] = None
    if 'Status_25_26' not in gdf.columns:
        gdf['Status_25_26'] = None

    # Get unique SMU layers needed
    smu_layers_needed = gdf['smu_layer'].dropna().unique()

    print(f"\n{programme_name}: Processing {len(smu_layers_needed)} unique SMU layers")

    # ===== SECTION 1: PROCESS EACH SMU LAYER =====
    for smu_layer in smu_layers_needed:
        print(f"  Processing {smu_layer}...")

        try:
            # Load the SMU layer from the file geodatabase
            smu_gdf = gpd.read_file(gdb_path, layer=smu_layer)

            # Get observations that should join with this SMU layer
            mask = gdf['smu_layer'] == smu_layer
            obs_for_this_smu = gdf[mask].copy()

            # STANDARD WITHIN JOIN
            joined = gpd.sjoin(
                obs_for_this_smu,
                smu_gdf[['operatingArea', 'Name', 'SMU_Name', 'Designation', 'Status_25_26', 'geometry']],
                how='left',
                predicate='within'
            )

            # Find the correct column names after the join
            field_map = {}
            for field in ['operatingArea', 'Name', 'SMU_Name', 'Designation', 'Status_25_26']:
                if f'{field}_right' in joined.columns:
                    field_map[field] = f'{field}_right'
                elif field in joined.columns:
                    field_map[field] = field

            print(f"    Field mapping: {field_map}")

            # Update main GeoDataFrame with matched observations
            matched_count = 0
            unmatched_indices = []
            
            for idx in joined.index:
                if pd.notna(joined.loc[idx, 'index_right']):
                    matched_count += 1
                    # Update each field using the correct column name
                    for field, col_name in field_map.items():
                        if pd.notna(joined.loc[idx, col_name]):
                            gdf.at[idx, field] = joined.loc[idx, col_name]
                else:
                    # Track unmatched observations for distance-based join
                    unmatched_indices.append(idx)

            print(f"    Matched {matched_count}/{len(obs_for_this_smu)} observations to SMU")

            # DISTANCE-BASED NEAREST JOIN FOR UNMATCHED OBSERVATIONS
            if len(unmatched_indices) > 0:
                print(f"    Attempting nearest join for {len(unmatched_indices)} unmatched observations (within {distance_threshold}m)...")
                
                unmatched_obs = obs_for_this_smu.loc[unmatched_indices].copy()
                
                # Find nearest SMU polygon for each unmatched observation
                nearest_matched = 0
                for idx in unmatched_indices:
                    point = unmatched_obs.loc[idx, 'geometry']
                    
                    # Calculate distance to each SMU polygon
                    smu_gdf['distance'] = smu_gdf.geometry.distance(point)
                    
                    # Get nearest polygon
                    nearest_idx = smu_gdf['distance'].idxmin()
                    nearest_distance = smu_gdf.loc[nearest_idx, 'distance']
                    
                    # Only join if within distance threshold
                    if nearest_distance <= distance_threshold:
                        nearest_matched += 1
                        # Update fields from nearest polygon
                        for field in ['operatingArea', 'Name', 'SMU_Name', 'Designation', 'Status_25_26']:
                            if field in smu_gdf.columns and pd.notna(smu_gdf.loc[nearest_idx, field]):
                                gdf.at[idx, field] = smu_gdf.loc[nearest_idx, field]
                
                if nearest_matched > 0:
                    print(f"    ✓ Matched {nearest_matched} observations via nearest join (within {distance_threshold}m)")
                else:
                    print(f"    No observations within {distance_threshold}m of SMU boundary")
                
                # Clean up temporary column
                if 'distance' in smu_gdf.columns:
                    smu_gdf.drop(columns=['distance'], inplace=True)

        except Exception as e:
            print(f"    Error loading {smu_layer}: {e}")
            import traceback
            traceback.print_exc()

    # ===== SECTION 2: SECONDARY FALLBACK (Progressive Containment only) =====
    if programme_name == 'Progressive Containment':
        no_smu_match = gdf['SMU_Name'].isna()
        target_species = ['Evergreen Buckthorn', 'Grey Willow']
        needs_pc_zone = no_smu_match & gdf['speciesName'].isin(target_species)

        if needs_pc_zone.sum() > 0:
            print(f"\n  Applying combined PC Mapped Zone + OperatingAreas for {needs_pc_zone.sum()} observations...")

            try:
                # Step 1: Read PC Mapped Zone from SDE using arcpy helper
                sde_feature_name = 'biosecurity.GISADMIN.Weeds_ProgressiveContainmentMappedZone'
                pc_zone_gdf = arc_sde_feature_to_gdf(sde_path, sde_feature_name)
                
                # CRITICAL: Filter by species to avoid overlapping polygons for different species
                # The PC zone has overlapping polygons for different species, so filter first
                obs_need_pc = gdf[needs_pc_zone].copy()
                
                # Get unique species in this batch (should only be Evergreen Buckthorn or Grey Willow)
                species_in_batch = obs_need_pc['speciesName'].unique()
                print(f"    Filtering PC zone for species: {species_in_batch}")
                
                # Filter PC zone to only include polygons for these species
                if 'Species' in pc_zone_gdf.columns:
                    pc_zone_gdf = pc_zone_gdf[pc_zone_gdf['Species'].isin(species_in_batch)]
                    print(f"    Filtered PC zone to {len(pc_zone_gdf)} polygons")
                else:
                    print(f"    WARNING: No 'Species' column in PC zone - cannot filter by species!")

                # Ensure CRS alignment
                if pc_zone_gdf.crs is not None and obs_need_pc.crs is not None and pc_zone_gdf.crs != obs_need_pc.crs:
                    obs_need_pc = obs_need_pc.to_crs(pc_zone_gdf.crs)
                    print("    Reprojected observations to PC zone CRS for spatial join.")
                elif pc_zone_gdf.crs is None:
                    print("    WARNING: pc_zone_gdf has no CRS; spatial join may be unreliable.")

                # Choose columns to join: prefer Label and geometry
                if 'Label' in pc_zone_gdf.columns:
                    join_cols = ['Label', 'geometry']
                else:
                    # fallback: any designation-like fields
                    candidates = [c for c in pc_zone_gdf.columns if c.lower().startswith('label') or 'designation' in c.lower()]
                    if candidates:
                        join_cols = [candidates[0], 'geometry']
                    else:
                        # If nothing plausible, just join geometry to inspect
                        join_cols = ['geometry']

                # STANDARD WITHIN JOIN
                joined_pc = gpd.sjoin(
                    obs_need_pc,
                    pc_zone_gdf[join_cols],
                    how='left',
                    predicate='within'
                )

                designation_candidates = [c for c in joined_pc.columns if (c.lower().startswith('label') or 'designation' in c.lower())]
                designation_col = next((c for c in designation_candidates if c.endswith('_right')), designation_candidates[0]) if designation_candidates else None

                print(f"    PC zone designation candidates: {designation_candidates}, chosen: {designation_col}")

                matched_mask_pc = joined_pc['index_right'].notna() if 'index_right' in joined_pc.columns else joined_pc.index.notna()
                matched_indices_pc = joined_pc.index[matched_mask_pc]

                # Track unmatched for distance join
                unmatched_pc_indices = []
                matched_pc = 0
                
                for idx in obs_need_pc.index:
                    if idx in matched_indices_pc and designation_col:
                        val = joined_pc.at[idx, designation_col]
                        if isinstance(val, (pd.Series, pd.DataFrame)):
                            try:
                                val = val.dropna().iloc[0] if not val.dropna().empty else None
                            except Exception:
                                val = None
                        if pd.notna(val):
                            gdf.at[idx, 'Designation'] = val
                            matched_pc += 1
                            continue
                    unmatched_pc_indices.append(idx)

                print(f"    Matched {matched_pc} observations to PC Mapped Zone (Designation)")

                # DISTANCE-BASED JOIN FOR UNMATCHED PC ZONE
                if len(unmatched_pc_indices) > 0:
                    print(f"    Attempting nearest join for {len(unmatched_pc_indices)} unmatched PC zone observations (within {distance_threshold}m)...")
                    
                    nearest_pc_matched = 0
                    for idx in unmatched_pc_indices:
                        if idx not in gdf.index:
                            continue
                        
                        point = gdf.loc[idx, 'geometry']
                        
                        # Calculate distance to each PC zone polygon
                        pc_zone_gdf['distance'] = pc_zone_gdf.geometry.distance(point)
                        nearest_idx = pc_zone_gdf['distance'].idxmin()
                        nearest_distance = pc_zone_gdf.loc[nearest_idx, 'distance']
                        
                        # Only join if within distance threshold
                        if nearest_distance <= distance_threshold:
                            if 'Label' in pc_zone_gdf.columns:
                                val = pc_zone_gdf.loc[nearest_idx, 'Label']
                                if pd.notna(val):
                                    gdf.at[idx, 'Designation'] = val
                                    nearest_pc_matched += 1
                    
                    if nearest_pc_matched > 0:
                        print(f"    ✓ Matched {nearest_pc_matched} PC zone observations via nearest join")
                    
                    # Clean up
                    if 'distance' in pc_zone_gdf.columns:
                        pc_zone_gdf.drop(columns=['distance'], inplace=True)

                # Step 2: Use OperatingAreas for operatingArea & Name
                operating_areas_gdf = gpd.read_file(gdb_path, layer='OperatingAreas')

                if operating_areas_gdf.crs != obs_need_pc.crs:
                    obs_for_oa = obs_need_pc.to_crs(operating_areas_gdf.crs)
                else:
                    obs_for_oa = obs_need_pc

                # STANDARD WITHIN JOIN
                joined_oa_pc = gpd.sjoin(
                    obs_for_oa,
                    operating_areas_gdf[['operatingArea', 'Name', 'geometry']],
                    how='left',
                    predicate='within'
                )

                oa_col = 'operatingArea_right' if 'operatingArea_right' in joined_oa_pc.columns else 'operatingArea'
                name_col = 'Name_right' if 'Name_right' in joined_oa_pc.columns else 'Name'

                print(f"    operatingArea candidates: chosen: {oa_col}")
                print(f"    Name candidates: chosen: {name_col}")

                # Track unmatched for distance join
                unmatched_oa_pc_indices = []
                matched_oa_pc = 0
                
                for idx in joined_oa_pc.index:
                    if idx not in gdf.index:
                        continue
                    
                    if pd.notna(joined_oa_pc.loc[idx, 'index_right']):
                        if oa_col and pd.notna(joined_oa_pc.loc[idx, oa_col]):
                            gdf.at[idx, 'operatingArea'] = joined_oa_pc.loc[idx, oa_col]
                        if name_col and pd.notna(joined_oa_pc.loc[idx, name_col]):
                            gdf.at[idx, 'Name'] = joined_oa_pc.loc[idx, name_col]
                        matched_oa_pc += 1
                    else:
                        unmatched_oa_pc_indices.append(idx)

                print(f"    Matched {matched_oa_pc} observations to OperatingAreas (operatingArea & Name)")

                # DISTANCE-BASED JOIN FOR UNMATCHED OPERATING AREAS
                if len(unmatched_oa_pc_indices) > 0:
                    print(f"    Attempting nearest join for {len(unmatched_oa_pc_indices)} unmatched OperatingAreas observations (within {distance_threshold}m)...")
                    
                    nearest_oa_pc_matched = 0
                    for idx in unmatched_oa_pc_indices:
                        if idx not in gdf.index:
                            continue
                        
                        point = gdf.loc[idx, 'geometry']
                        
                        # Calculate distance to each operating area polygon
                        operating_areas_gdf['distance'] = operating_areas_gdf.geometry.distance(point)
                        nearest_idx = operating_areas_gdf['distance'].idxmin()
                        nearest_distance = operating_areas_gdf.loc[nearest_idx, 'distance']
                        
                        # Only join if within distance threshold
                        if nearest_distance <= distance_threshold:
                            if 'operatingArea' in operating_areas_gdf.columns:
                                gdf.at[idx, 'operatingArea'] = operating_areas_gdf.loc[nearest_idx, 'operatingArea']
                            if 'Name' in operating_areas_gdf.columns:
                                gdf.at[idx, 'Name'] = operating_areas_gdf.loc[nearest_idx, 'Name']
                            nearest_oa_pc_matched += 1
                    
                    if nearest_oa_pc_matched > 0:
                        print(f"    ✓ Matched {nearest_oa_pc_matched} OperatingAreas observations via nearest join")
                    
                    # Clean up
                    if 'distance' in operating_areas_gdf.columns:
                        operating_areas_gdf.drop(columns=['distance'], inplace=True)
                
                # Final check: Ensure all Evergreen Buckthorn/Grey Willow have Designation
                still_no_designation = needs_pc_zone & gdf['Designation'].isna()
                if still_no_designation.sum() > 0:
                    print(f"    WARNING: {still_no_designation.sum()} Evergreen Buckthorn/Grey Willow observations still without Designation")
                    print(f"    These observations may be >100m from PC zone boundary - consider increasing distance_threshold")

            except Exception as e:
                print(f"    Error with PC zone + OperatingAreas fallback: {e}")
                import traceback
                traceback.print_exc()

    # ===== SECTION 3: TERTIARY FALLBACK - For remaining observations without SMU match =====
    no_smu_match = gdf['SMU_Name'].isna()
    # Exclude Evergreen Buckthorn and Grey Willow (which were handled above)
    if programme_name == 'Progressive Containment':
        target_species = ['Evergreen Buckthorn', 'Grey Willow']
        no_smu_match = no_smu_match & ~gdf['speciesName'].isin(target_species)

    if no_smu_match.sum() > 0:
        print(f"\n  Applying OperatingAreas fallback for {no_smu_match.sum()} observations...")

        try:
            # For Progressive Containment: Also need to get Designation from PC Mapped Zone
            if programme_name == 'Progressive Containment':
                print(f"    Also joining with PC Mapped Zone for Designation...")
                
                try:
                    sde_feature_name = 'biosecurity.GISADMIN.Weeds_ProgressiveContainmentMappedZone'
                    pc_zone_gdf = arc_sde_feature_to_gdf(sde_path, sde_feature_name)
                    
                    obs_no_smu = gdf[no_smu_match].copy()
                    
                    # CRITICAL: Filter by species to avoid overlapping polygons
                    species_in_batch = obs_no_smu['speciesName'].unique()
                    print(f"    Filtering PC zone for species: {species_in_batch}")
                    
                    if 'Species' in pc_zone_gdf.columns:
                        pc_zone_gdf = pc_zone_gdf[pc_zone_gdf['Species'].isin(species_in_batch)]
                        print(f"    Filtered PC zone to {len(pc_zone_gdf)} polygons")
                    
                    # Ensure CRS alignment
                    if pc_zone_gdf.crs is not None and obs_no_smu.crs is not None and pc_zone_gdf.crs != obs_no_smu.crs:
                        obs_no_smu = obs_no_smu.to_crs(pc_zone_gdf.crs)
                    
                    # Choose columns to join: prefer Label and geometry
                    if 'Label' in pc_zone_gdf.columns:
                        join_cols = ['Label', 'geometry']
                    else:
                        # fallback: any designation-like fields
                        candidates = [c for c in pc_zone_gdf.columns if c.lower().startswith('label') or 'designation' in c.lower()]
                        if candidates:
                            join_cols = [candidates[0], 'geometry']
                        else:
                            # If nothing plausible, just join geometry to inspect
                            join_cols = ['geometry']
                    
                    # STANDARD WITHIN JOIN
                    joined_pc_fallback = gpd.sjoin(
                        obs_no_smu,
                        pc_zone_gdf[join_cols],
                        how='left',
                        predicate='within'
                    )
                    
                    designation_candidates = [c for c in joined_pc_fallback.columns if (c.lower().startswith('label') or 'designation' in c.lower())]
                    designation_col = next((c for c in designation_candidates if c.endswith('_right')), designation_candidates[0]) if designation_candidates else None
                    
                    matched_pc_fallback = 0
                    unmatched_pc_fallback_indices = []
                    
                    for idx in obs_no_smu.index:
                        if idx not in gdf.index:
                            continue
                        
                        # Handle potential Series from index_right
                        matched_to_pc = False
                        if 'index_right' in joined_pc_fallback.columns:
                            idx_right_val = joined_pc_fallback.loc[idx, 'index_right']
                            # Convert the series to a single value if needed
                            if isinstance(idx_right_val, pd.Series):
                                idx_right_val = idx_right_val.iloc[0] if not idx_right_val.empty else None
                            matched_to_pc = pd.notna(idx_right_val)
                        
                        if matched_to_pc:
                            if designation_col and designation_col in joined_pc_fallback.columns:
                                val = joined_pc_fallback.loc[idx, designation_col]
                                if isinstance(val, (pd.Series, pd.DataFrame)):
                                    try:
                                        val = val.dropna().iloc[0] if not val.dropna().empty else None
                                    except Exception:
                                        val = None
                                if pd.notna(val):
                                    gdf.at[idx, 'Designation'] = val
                                    matched_pc_fallback += 1
                                    continue
                        unmatched_pc_fallback_indices.append(idx)
                    
                    print(f"    Matched {matched_pc_fallback} observations to PC Mapped Zone")
                    
                    # DISTANCE-BASED JOIN FOR UNMATCHED PC ZONE
                    if len(unmatched_pc_fallback_indices) > 0:
                        print(f"    Attempting nearest join for {len(unmatched_pc_fallback_indices)} unmatched PC zone observations (within {distance_threshold}m)...")
                        
                        nearest_pc_fallback_matched = 0
                        for idx in unmatched_pc_fallback_indices:
                            if idx not in gdf.index:
                                continue
                            
                            point = gdf.loc[idx, 'geometry']
                            
                            # Calculate distance to each PC zone polygon
                            pc_zone_gdf['distance'] = pc_zone_gdf.geometry.distance(point)
                            nearest_idx = pc_zone_gdf['distance'].idxmin()
                            nearest_distance = pc_zone_gdf.loc[nearest_idx, 'distance']
                            
                            # Only join if within distance threshold
                            if nearest_distance <= distance_threshold:
                                if 'Label' in pc_zone_gdf.columns:
                                    val = pc_zone_gdf.loc[nearest_idx, 'Label']
                                    if pd.notna(val):
                                        gdf.at[idx, 'Designation'] = val
                                        nearest_pc_fallback_matched += 1
                        
                        if nearest_pc_fallback_matched > 0:
                            print(f"    ✓ Matched {nearest_pc_fallback_matched} PC zone observations via nearest join")
                        
                        # Clean up
                        if 'distance' in pc_zone_gdf.columns:
                            pc_zone_gdf.drop(columns=['distance'], inplace=True)
                    
                except Exception as e:
                    print(f"    Error with PC Mapped Zone in tertiary fallback: {e}")
                    import traceback
                    traceback.print_exc()
            
            # Load OperatingAreas layer (for all programmes)
            operating_areas_gdf = gpd.read_file(gdb_path, layer='OperatingAreas')

            # Get observations without SMU match
            obs_no_smu = gdf[no_smu_match].copy()

            # STANDARD WITHIN JOIN
            joined_oa = gpd.sjoin(
                obs_no_smu,
                operating_areas_gdf[['operatingArea', 'Name', 'geometry']],
                how='left',
                predicate='within'
            )

            # Track unmatched for nearest join
            unmatched_oa_indices = []
            matched_oa = 0
            
            for idx in joined_oa.index:
                if pd.notna(joined_oa.loc[idx, 'index_right']):
                    matched_oa += 1
                    # Update operatingArea
                    if 'operatingArea_right' in joined_oa.columns and pd.notna(joined_oa.loc[idx, 'operatingArea_right']):
                        gdf.at[idx, 'operatingArea'] = joined_oa.loc[idx, 'operatingArea_right']
                    # Update Name (operator name)
                    if 'Name_right' in joined_oa.columns and pd.notna(joined_oa.loc[idx, 'Name_right']):
                        gdf.at[idx, 'Name'] = joined_oa.loc[idx, 'Name_right']
                    # Set Designation to AMZ for Eradication/Exclusion (only if not already set)
                    if programme_name in ['Eradication', 'Exclusion'] and pd.isna(gdf.at[idx, 'Designation']):
                        gdf.at[idx, 'Designation'] = 'AMZ'
                else:
                    unmatched_oa_indices.append(idx)

            print(f"    Matched {matched_oa} observations to OperatingAreas")

            # DISTANCE-BASED JOIN FOR UNMATCHED OPERATING AREAS
            if len(unmatched_oa_indices) > 0:
                print(f"    Attempting nearest join for {len(unmatched_oa_indices)} unmatched observations (within {distance_threshold}m)...")
                
                nearest_oa_matched = 0
                for idx in unmatched_oa_indices:
                    if idx not in gdf.index:
                        continue
                    
                    point = gdf.loc[idx, 'geometry']
                    
                    # Calculate distance to each operating area polygon
                    operating_areas_gdf['distance'] = operating_areas_gdf.geometry.distance(point)
                    nearest_idx = operating_areas_gdf['distance'].idxmin()
                    nearest_distance = operating_areas_gdf.loc[nearest_idx, 'distance']
                    
                    # Only join if within distance threshold
                    if nearest_distance <= distance_threshold:
                        nearest_oa_matched += 1
                        if 'operatingArea' in operating_areas_gdf.columns and pd.notna(operating_areas_gdf.loc[nearest_idx, 'operatingArea']):
                            gdf.at[idx, 'operatingArea'] = operating_areas_gdf.loc[nearest_idx, 'operatingArea']
                        if 'Name' in operating_areas_gdf.columns and pd.notna(operating_areas_gdf.loc[nearest_idx, 'Name']):
                            gdf.at[idx, 'Name'] = operating_areas_gdf.loc[nearest_idx, 'Name']
                        if programme_name in ['Eradication', 'Exclusion'] and pd.isna(gdf.at[idx, 'Designation']):
                            gdf.at[idx, 'Designation'] = 'AMZ'
                
                if nearest_oa_matched > 0:
                    print(f"    ✓ Matched {nearest_oa_matched} observations via nearest join")
                
                # Clean up
                if 'distance' in operating_areas_gdf.columns:
                    operating_areas_gdf.drop(columns=['distance'], inplace=True)

        except Exception as e:
            print(f"    Error with OperatingAreas fallback: {e}")
            import traceback
            traceback.print_exc()

    # Final fallback: Set Designation = 'AMZ' for any remaining Eradication/Exclusion
    if programme_name in ['Eradication', 'Exclusion']:
        still_no_designation = gdf['Designation'].isna()
        if still_no_designation.sum() > 0:
            gdf.loc[still_no_designation, 'Designation'] = 'AMZ'
            print(f"  Set {still_no_designation.sum()} remaining observations to Designation='AMZ'")

    return gdf

# Perform spatial joins
print("Starting spatial joins...")
gdf_progressive_joined = spatial_join_with_smu(gdf_progressive.copy(), 'Progressive Containment')
gdf_eradication_joined = spatial_join_with_smu(gdf_eradication.copy(), 'Eradication')
gdf_exclusion_joined = spatial_join_with_smu(gdf_exclusion.copy(), 'Exclusion')

print("\n=== Summary ===")
print(f"Progressive: {gdf_progressive_joined['SMU_Name'].notna().sum()}/{len(gdf_progressive_joined)} matched to SMU")
print(f"Eradication: {gdf_eradication_joined['SMU_Name'].notna().sum()}/{len(gdf_eradication_joined)} matched to SMU")
print(f"Exclusion: {gdf_exclusion_joined['SMU_Name'].notna().sum()}/{len(gdf_exclusion_joined)} matched to SMU")

print(f"\nProgressive: {gdf_progressive_joined['operatingArea'].notna().sum()}/{len(gdf_progressive_joined)} have operatingArea")
print(f"Eradication: {gdf_eradication_joined['operatingArea'].notna().sum()}/{len(gdf_eradication_joined)} have operatingArea")
print(f"Exclusion: {gdf_exclusion_joined['operatingArea'].notna().sum()}/{len(gdf_exclusion_joined)} have operatingArea")

print(f"\nProgressive: {gdf_progressive_joined['Designation'].notna().sum()}/{len(gdf_progressive_joined)} have Designation")
print(f"Eradication: {gdf_eradication_joined['Designation'].notna().sum()}/{len(gdf_eradication_joined)} have Designation")
print(f"Exclusion: {gdf_exclusion_joined['Designation'].notna().sum()}/{len(gdf_exclusion_joined)} have Designation")

Starting spatial joins...

Progressive Containment: Processing 6 unique SMU layers
  Processing SMU_Mothplant...
    Field mapping: {'operatingArea': 'operatingArea_right', 'Name': 'Name_right', 'SMU_Name': 'SMU_Name_right', 'Designation': 'Designation_right', 'Status_25_26': 'Status_25_26_right'}
    Matched 8/8 observations to SMU
  Processing SMU_Old_mans_beard...
    Field mapping: {'operatingArea': 'operatingArea_right', 'Name': 'Name_right', 'SMU_Name': 'SMU_Name_right', 'Designation': 'Designation_right', 'Status_25_26': 'Status_25_26_right'}
    Matched 173/173 observations to SMU
  Processing SMU_Pest_conifers...
    Field mapping: {'operatingArea': 'operatingArea_right', 'Name': 'Name_right', 'SMU_Name': 'SMU_Name_right', 'Designation': 'Designation_right', 'Status_25_26': 'Status_25_26_right'}
    Matched 15/16 observations to SMU
    Attempting nearest join for 1 unmatched observations (within 100m)...
    No observations within 100m of SMU boundary
  Processing SMU_Bonesee

In [20]:
# Check what data we have now
print("=== Progressive Containment Sample ===")
print(gdf_progressive_joined[['taxon_name', 'user_name', 'operatingArea', 'Name', 'SMU_Name', 'Designation', 'flowers_fruits']].head(10))

print("\n=== Eradication Sample ===")
print(gdf_eradication_joined[['taxon_name', 'user_name', 'operatingArea', 'Name', 'SMU_Name', 'Designation', 'flowers_fruits']].head(10))

print("\n=== Exclusion Sample ===")
print(gdf_exclusion_joined[['taxon_name', 'user_name', 'operatingArea', 'Name', 'SMU_Name', 'Designation', 'flowers_fruits']].head(10))

# Check for observations WITH SMU_Name
print("\n=== Examples WITH SMU_Name ===")
print("Progressive with SMU:")
print(gdf_progressive_joined[gdf_progressive_joined['SMU_Name'].notna()][['taxon_name', 'operatingArea', 'Name', 'SMU_Name', 'Designation']].head(5))

print("\nEradication with SMU:")
print(gdf_eradication_joined[gdf_eradication_joined['SMU_Name'].notna()][['taxon_name', 'operatingArea', 'Name', 'SMU_Name', 'Designation']].head(5))

=== Progressive Containment Sample ===
            taxon_name    user_name     operatingArea                Name  \
1   Araujia sericifera   Colin Ogle         Whanganui       Robbie Sicely   
3     Clematis vitalba                        Taihape  Malinda Matthewson   
4     Clematis vitalba  Hēmi Tūtiri         Whanganui       Robbie Sicely   
5        Salix cinerea         None           Tararua          Jack Keast   
6       Pinus contorta                      Tongariro       Kelsi Hoggard   
7     Clematis vitalba               Palmerston North        Rory Johnson   
8   Araujia sericifera   Colin Ogle         Whanganui       Robbie Sicely   
9     Clematis vitalba         None  Palmerston North        Rory Johnson   
11   Rhamnus alaternus                      Whanganui       Robbie Sicely   
13    Clematis vitalba         None           Tararua          Jack Keast   

     SMU_Name Designation flowers_fruits  
1      Active         AMZ           None  
3        GNPZ        GNPZ  

## Spatial Join — Managed Pest Plant Sites

Intersects Eradication and Progressive Containment observations against the 
BioS Pest Plant Sites hosted feature layer on AGOL.

Added two new fields:
- `is_in_site` — `'Y'` if the observation falls within a managed site, otherwise `'N'`
- `BaseSiteID` — the BaseSiteID of the intersected site, or `None` if no match

**Programme mapping:**
- Eradication observations → sites where `projectType = 'Eradication'`
- Progressive Containment observations → sites where `projectType = 'Progressive Containment - Mapped'`

In [37]:
# ============================================================================
# Spatial Join — Managed Pest Plant Sites
# ============================================================================
# Fetches BioS_Pest_Plants_Sites from AGOL and intersects with
# Eradication and Progressive Containment observations.
# Filters to active sites only (activeSite = 'Y').
# Matches on speciesID field to avoid domain resolution issues.
# Adds: is_in_site (Y/N) and BaseSiteID fields.
# A 10m buffer is applied to sites to catch observations just outside boundaries.
# ============================================================================

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import arcpy
from shapely.geometry import Polygon
import geopandas as gpd
import pandas as pd

PEST_PLANT_SITES_URL = "https://services1.arcgis.com/VuN78wcRdq1Oj69W/arcgis/rest/services/BioS_Pest_Plants_Sites/FeatureServer/0"

SITE_BUFFER_METRES = 10

# Maps site speciesID code → iNat speciesName
# Used to match site species to iNat observation species
SITE_SPECIES_CODE_TO_INAT = {
    # Eradication
    'AFG': 'African Feathergrass',
    'ALW': 'Alligator Weed',
    'ARH': 'Arrowhead',
    'BPF': 'Blue Passionflower',
    'CAL': 'Climbing Alstroemeria',
    'CBS': 'Cathedral Bells',
    'CHR': 'Chilean Rhubarb',
    'CPS': 'Chinese Pennisetum',
    'CSB': 'Climbing Spindleberry',
    'HBA': 'Himalayan Balsam',
    'KNW': 'Japanese Knotweed',
    'NAS': 'Nassella Tussock',
    'PLS': 'Purple Loosestrife',
    'QPL': 'Queensland Poplar',
    'RCY': 'Rum Cherry',
    'SNT': 'Senegal Tea',
    'SPT': 'Spartina',
    'WNS': 'Woolly Nightshade',
    # Progressive Containment
    'BAP': 'Banana Passionfruit',
    'BSD': 'Boneseed',
    'CON': 'Pest Conifers',
    'DBY': "Darwin's Barberry",
    'DMP': 'Pest Conifers',
    'EBT': 'Evergreen Buckthorn',
    'GWO': 'Grey Willow',
    'MPT': 'Moth Plant',
    'OMB': "Old Man's Beard",
    'SCP': 'Pest Conifers',
}

PROGRAMME_TO_PROJECT_TYPE = {
    'Eradication':             'Eradication',
    'Progressive Containment': 'Progressive Containment - Mapped'
}


def fetch_pest_plant_sites(layer_url, project_type_filter):
    """
    Fetch active BioS Pest Plant Sites using arcpy (authenticated via ArcGIS Pro).
    Filters to activeSite = 'Y' and the given projectType.
    Maps speciesID codes to iNat speciesName values.
    Returns a GeoDataFrame in NZTM2000 (EPSG:2193), or None on failure.
    """
    try:
        print(f"  Fetching active sites for projectType = '{project_type_filter}'...")
        where = f"projectType = '{project_type_filter}' AND activeSite = 'Y'"

        rows = []
        with arcpy.da.SearchCursor(
            layer_url,
            ['SHAPE@', 'BaseSiteID', 'specieID'],
            where_clause=where
        ) as cursor:
            for row in cursor:
                shape, base_site_id, species_id = row
                if shape is None:
                    continue
                try:
                    # Convert arcpy polygon to shapely
                    wkb = bytes(shape.WKB)
                    import shapely.wkb
                    geom = shapely.wkb.loads(wkb)
                    if not geom.is_valid:
                        geom = geom.buffer(0)

                    # Map species code to iNat species name
                    inat_species = SITE_SPECIES_CODE_TO_INAT.get(species_id)

                    rows.append({
                        'BaseSiteID':   base_site_id,
                        'speciesID':    species_id,
                        'inat_species': inat_species,
                        'geometry':     geom
                    })
                except Exception:
                    continue

        if not rows:
            print(f"  ⚠ No valid features found")
            return None

        sites_gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:2193')
        sites_gdf = sites_gdf[sites_gdf.geometry.is_valid & sites_gdf.geometry.notna()]
        print(f"  ✓ Fetched {len(sites_gdf)} active sites")
        return sites_gdf

    except Exception as e:
        print(f"  ✗ Error fetching sites: {e}")
        import traceback; traceback.print_exc()
        return None


def add_site_fields(obs_gdf, sites_gdf, programme_name):
    """
    Spatially intersect observations with pest plant sites (with 10m buffer).
    Only matches where the site speciesID maps to the same iNat speciesName
    as the observation — prevents cross-species false matches.
    Adds is_in_site ('Y'/'N') and BaseSiteID (value or None).
    """
    obs_gdf = obs_gdf.copy()
    obs_gdf['is_in_site'] = 'N'
    obs_gdf['BaseSiteID'] = None

    if sites_gdf is None or len(sites_gdf) == 0:
        print(f"  ⚠ No sites available for {programme_name} — all set to is_in_site='N'")
        return obs_gdf

    if obs_gdf.crs != sites_gdf.crs:
        obs_gdf = obs_gdf.to_crs(sites_gdf.crs)

    # Apply buffer to catch observations just outside site boundaries
    sites_buffered = sites_gdf.copy()
    sites_buffered['geometry'] = sites_gdf.geometry.buffer(SITE_BUFFER_METRES)

    # Spatial join — get all candidate site matches per observation
    joined = gpd.sjoin(
        obs_gdf[['speciesName', 'geometry']],
        sites_buffered[['BaseSiteID', 'inat_species', 'geometry']],
        how='left',
        predicate='within'
    )

    match_count = 0
    joined_deduped = joined[~joined.index.duplicated(keep='first')]

    for idx in joined_deduped.index:
        site_species = joined_deduped.loc[idx, 'inat_species']
        obs_species  = joined_deduped.loc[idx, 'speciesName']
        base_id      = joined_deduped.loc[idx, 'BaseSiteID']

        if pd.notna(base_id) and pd.notna(site_species) and site_species == obs_species:
            obs_gdf.at[idx, 'is_in_site'] = 'Y'
            obs_gdf.at[idx, 'BaseSiteID'] = str(base_id)
            match_count += 1

    print(f"  ✓ {programme_name}: {match_count}/{len(obs_gdf)} observations matched to an active managed site (species-matched)")
    return obs_gdf


print("="*70)
print("SPATIAL JOIN — MANAGED PEST PLANT SITES")
print("="*70)
print(f"Site buffer: {SITE_BUFFER_METRES}m | Active sites only | Species-matched")

# Fetch sites for each programme
print("\n[1/2] Eradication sites...")
erad_sites_gdf = fetch_pest_plant_sites(PEST_PLANT_SITES_URL, PROGRAMME_TO_PROJECT_TYPE['Eradication'])

print("\n[2/2] Progressive Containment sites...")
prog_sites_gdf = fetch_pest_plant_sites(PEST_PLANT_SITES_URL, PROGRAMME_TO_PROJECT_TYPE['Progressive Containment'])

# Apply site intersection
print("\nApplying site intersection...")
print("  Eradication:")
gdf_eradication_joined = add_site_fields(gdf_eradication_joined, erad_sites_gdf, 'Eradication')
print("  Progressive Containment:")
gdf_progressive_joined = add_site_fields(gdf_progressive_joined, prog_sites_gdf, 'Progressive Containment')

# Summary
print("\n" + "="*70)
print("PEST PLANT SITES JOIN SUMMARY")
print("="*70)
erad_in = (gdf_eradication_joined['is_in_site'] == 'Y').sum()
prog_in  = (gdf_progressive_joined['is_in_site'] == 'Y').sum()
print(f"  Eradication:             {erad_in}/{len(gdf_eradication_joined)} matched to an active site (species-matched, within {SITE_BUFFER_METRES}m)")
print(f"  Progressive Containment: {prog_in}/{len(gdf_progressive_joined)} matched to an active site (species-matched, within {SITE_BUFFER_METRES}m)")
print(f"\n  Observations where is_in_site = 'N' may indicate a new infestation.")
print("="*70)

SPATIAL JOIN — MANAGED PEST PLANT SITES
Site buffer: 10m | Active sites only | Species-matched

[1/2] Eradication sites...
  Fetching active sites for projectType = 'Eradication'...
  ✓ Fetched 1956 active sites

[2/2] Progressive Containment sites...
  Fetching active sites for projectType = 'Progressive Containment - Mapped'...
  ✓ Fetched 2844 active sites

Applying site intersection...
  Eradication:
  ✓ Eradication: 46/137 observations matched to an active managed site (species-matched)
  Progressive Containment:
  ✓ Progressive Containment: 37/377 observations matched to an active managed site (species-matched)

PEST PLANT SITES JOIN SUMMARY
  Eradication:             46/137 matched to an active site (species-matched, within 10m)
  Progressive Containment: 37/377 matched to an active site (species-matched, within 10m)

  Observations where is_in_site = 'N' may indicate a new infestation.


## Export Joined GeoDataFrames to Feature Classes

In [38]:
# Output GDB path
# Output GDB path from config — not hardcoded
output_gdb = config.OUTPUT_GDB

# Prepare GeoDataFrame for export
def prepare_for_export(gdf):
    """Convert list and dict columns to strings for GDB export"""
    gdf_export = gdf.copy()
    
    # Photo URLs are already in separate columns as strings, no conversion needed
    # Ensure no NaN values in photoURL columns
    for col in ['photoURL_1', 'photoURL_2', 'photoURL_3']:
        if col in gdf_export.columns:
            gdf_export[col] = gdf_export[col].fillna('')
    
    # Convert obs_fields dict to string
    if 'obs_fields' in gdf_export.columns:
        gdf_export['obs_fields'] = gdf_export['obs_fields'].apply(
            lambda x: str(x) if x else None
        )
    
    # Drop the smu_layer column
    if 'smu_layer' in gdf_export.columns:
        gdf_export = gdf_export.drop(columns=['smu_layer'])
    
    return gdf_export 

# Function to set field aliases after export
def set_field_aliases(gdb_path, fc_name, alias_dict):
    """Set field aliases in the feature class"""
    import arcpy
    fc_path = os.path.join(gdb_path, fc_name)
    
    for field_name, alias_name in alias_dict.items():
        try:
            arcpy.management.AlterField(fc_path, field_name, new_field_alias=alias_name)
        except:
            pass 


# Define field aliases
field_aliases = {
    'observation_url': 'iNaturalist Observation URL Link',
    'taxon_name': 'Taxon Name',
    'place_guess': 'Location',
    'programme': 'RPMP Programme',
    'observed_on': 'Observation Date',
    'user_login': 'Observer User Name',
    'Name': 'Operator Name',  
    'operatingArea': 'Operating Area',
    'SMU_Name': 'SMU Name',
    'Status_25_26': 'Status 25-26',
    'quality_grade': 'Observation Quality Grade',
    'flowers_fruits': 'Flowers or Fruit',
    'description': 'Description',
    'is_in_site': 'In Managed Site',
    'BaseSiteID': 'Base Site ID'
}

# Export each programme's GeoDataFrame to the output GDB
print("Exporting feature classes to GDB...")

try:
    # Progressive Containment
    gdf_prog_export = prepare_for_export(gdf_progressive_joined)
    gdf_prog_export.to_file(
        output_gdb, 
        layer='iNat_ProgressiveContainment', 
        driver='OpenFileGDB'
    )

    set_field_aliases(output_gdb, 'iNat_ProgressiveContainment', field_aliases)  
    print(f"✓ Exported iNat_ProgressiveContainment ({len(gdf_prog_export)} features)")
    
    # Eradication
    gdf_erad_export = prepare_for_export(gdf_eradication_joined)
    gdf_erad_export.to_file(
        output_gdb, 
        layer='iNat_Eradication', 
        driver='OpenFileGDB'
    )
    set_field_aliases(output_gdb, 'iNat_Eradication', field_aliases) 
    print(f"✓ Exported iNat_Eradication ({len(gdf_erad_export)} features)")
    
    
    # Exclusion
    gdf_excl_export = prepare_for_export(gdf_exclusion_joined)
    gdf_excl_export.to_file(
        output_gdb, 
        layer='iNat_Exclusion', 
        driver='OpenFileGDB'
    )

    set_field_aliases(output_gdb, 'iNat_Exclusion', field_aliases)  
    print(f"✓ Exported iNat_Exclusion ({len(gdf_excl_export)} features)")
    
    print(f"\n✓ All feature classes exported successfully to:")
    print(f"  {output_gdb}")
    
except Exception as e:
    print(f"✗ Error exporting feature classes: {e}")
    import traceback
    traceback.print_exc()

Exporting feature classes to GDB...
✓ Exported iNat_ProgressiveContainment (377 features)
✓ Exported iNat_Eradication (137 features)
✓ Exported iNat_Exclusion (3 features)

✓ All feature classes exported successfully to:
  \\gisdata\GIS\Department\Environmental_Management\Biodiversity\BioData\Biosecurity\iNaturalist\iNat_Biosecurity_Plants.gdb


## Export Combined Layer (All Programmes)

In [39]:
# Create a single combined feature class with all observations
gdf_all = pd.concat([
    gdf_progressive_joined,
    gdf_eradication_joined,
    gdf_exclusion_joined
], ignore_index=True)

try:
    # Prepare for export (convert lists/dicts to strings)
    gdf_all_export = prepare_for_export(gdf_all)
    
    gdf_all_export.to_file(
        output_gdb, 
        layer='iNat_AllProgrammes', 
        driver='OpenFileGDB'
    )

    set_field_aliases(output_gdb, 'iNat_AllProgrammes', field_aliases)  
    print(f"✓ Exported iNat_AllProgrammes ({len(gdf_all_export)} features)")

except Exception as e:
    print(f"✗ Error exporting combined layer: {e}")
    import traceback
    traceback.print_exc()

✓ Exported iNat_AllProgrammes (517 features)


## Track New Observations and Prepare Notifications

In [21]:
# Path to store previously processed observation IDs
# Tracking file path from config — not hardcoded
tracking_file = config.TRACKING_FILE

def load_previous_observations(file_path):
    """Load previously processed observation IDs from file"""
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
                print(f"✓ Loaded {len(data.get('observation_ids', []))} previously processed observations")
                print(f"  Last run: {data.get('last_run', 'Unknown')}")
                return set(data.get('observation_ids', []))
        except Exception as e:
            print(f"Error loading tracking file: {e}")
            print("  Starting fresh - all observations will be considered 'new'")
            return set()
    else:
        print("No tracking file found - first run, all observations are 'new'")
        return set()

def save_current_observations(file_path, observation_ids):
    """Save current observation IDs to file for next run"""
    try:
        data = {
            'last_run': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_observations': len(observation_ids),
            'observation_ids': list(observation_ids)
        }
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"✓ Saved {len(observation_ids)} observation IDs for next run")
        return True
    except Exception as e:
        print(f"✗ Error saving tracking file: {e}")
        return False

# Load previous observation IDs
previous_obs_ids = load_previous_observations(tracking_file)

# Get current observation IDs from the combined dataset
current_obs_ids = set(gdf_all['id'].tolist())

# Identify NEW observations (in current but not in previous)
new_obs_ids = current_obs_ids - previous_obs_ids

print(f"\n{'='*70}")
print("OBSERVATION TRACKING SUMMARY")
print(f"{'='*70}")
print(f"Previous observations: {len(previous_obs_ids)}")
print(f"Current observations:  {len(current_obs_ids)}")
print(f"New observations:      {len(new_obs_ids)}")
print(f"{'='*70}\n")

# Filter for NEW observations in priority programmes (Eradication & Exclusion)
if len(new_obs_ids) > 0:
    # Get new observations from the combined dataset
    new_observations = gdf_all[gdf_all['id'].isin(new_obs_ids)].copy()
    
    # Filter for Eradication and Exclusion programmes
    priority_new_obs = new_observations[
        new_observations['programme'].isin(['Eradication', 'Exclusion'])
    ].copy()
    
    print(f"NEW OBSERVATIONS BY PROGRAMME:")
    for programme in ['Progressive Containment', 'Eradication', 'Exclusion']:
        count = len(new_observations[new_observations['programme'] == programme])
        if count > 0:
            print(f"  {programme}: {count}")
    
    if len(priority_new_obs) > 0:
        print(f"\nPRIORITY: {len(priority_new_obs)} new Eradication/Exclusion observations require notification")
        
        # Group by operating area and staff member for notifications
        notification_summary = priority_new_obs.groupby(['operatingArea', 'Name', 'programme']).agg({
            'id': 'count',
            'taxon_name': lambda x: ', '.join(x.unique()),
            'speciesName': lambda x: ', '.join(x.dropna().unique()) if x.notna().any() else 'Unknown'
        }).reset_index()
        
        notification_summary.columns = ['Operating Area', 'Staff Member', 'Programme', 'Count', 'Scientific Names', 'Common Names']
        
        print("\nNOTIFICATION SUMMARY (by staff member):")
        print(notification_summary.to_string(index=False))
        
        # Store detailed notification data for email sending
        notification_data = []
        for _, row in priority_new_obs.iterrows():
            notification_data.append({
                'observation_id': row['id'],
                'observation_url': row.get('observation_url', f"https://www.inaturalist.org/observations/{row['id']}"),
                'programme': row['programme'],
                'taxon_name': row['taxon_name'],
                'species_name': row.get('speciesName', 'Unknown'),
                'observed_on': row['observed_on'],
                'operating_area': row.get('operatingArea', 'Unknown'),
                'staff_member': row.get('Name', 'Unassigned'),
                'latitude': row['latitude'],
                'longitude': row['longitude'],
                'quality_grade': row['quality_grade'],
                'user_name': row['user_name'],
                'description': row.get('description', ''),
                'flowers_fruits': row.get('flowers_fruits', ''),
                'photoURL_1': row.get('photoURL_1', '')
            })
        
        # Save notification data to JSON for email script to use
        notification_file = config.NOTIFICATION_FILE
        try:
            with open(notification_file, 'w') as f:
                json.dump({
                    'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'total_notifications': len(notification_data),
                    'notifications': notification_data
                }, f, indent=2)
            print(f"\n✓ Notification data saved to: {notification_file}")
            print(f"  Ready for email notification script")
        except Exception as e:
            print(f"\n✗ Error saving notification data: {e}")
    else:
        print("\n✓ No new Eradication/Exclusion observations - no notifications needed")
        
        # Clear any pending notifications file
        notification_file = config.NOTIFICATION_FILE
        if os.path.exists(notification_file):
            try:
                os.remove(notification_file)
                print("  Cleared previous notification file")
            except:
                pass
else:
    print("✓ No new observations since last run")

✓ Loaded 493 previously processed observations
  Last run: 2026-01-15 14:37:30

OBSERVATION TRACKING SUMMARY
Previous observations: 493
Current observations:  493
New observations:      0

✓ No new observations since last run


## Update Hosted Feature Layer on ArcGIS Online

In [40]:
# Initialise ArcGIS Online connection
print("Connecting to ArcGIS Online...")
gis = GIS("pro")
print(f"✓ Connected as: {gis.properties.user.username}")

# Configuration
# Service item ID from config — not hardcoded
SERVICE_ITEM_ID = config.AGOL_SERVICE_ITEM_ID

def update_hosted_service(item_id, gdb_path):
    """
    Update hosted feature service by deleting all features 
    and appending fresh data from GDB. Works with sync enabled.
    """
    try:
        print(f"\n{'='*70}")
        print("UPDATING HOSTED FEATURE SERVICE")
        print(f"{'='*70}\n")
        
        # Get the hosted feature layer item
        item = gis.content.get(item_id)
        
        if item is None:
            print(f"✗ Error: Could not find item with ID {item_id}")
            return False
        
        print(f"Found service: {item.title}")
        
        # Get feature layer collection
        from arcgis.features import FeatureLayerCollection
        flc = FeatureLayerCollection.fromitem(item)
        
        print(f"\nService has {len(flc.layers)} layers")
        
        # Map GDB layer names to service layer names
        layer_mapping = {
            'iNat_AllProgrammes': 'iNat_AllProgrammes',
            'iNat_Eradication': 'iNat_Eradication',
            'iNat_ProgressiveContainment': 'iNat_ProgressiveContainment',
            'iNat_Exclusion': 'iNat_Exclusion'
        }
        
        # Process each layer
        for gdb_layer_name, service_layer_name in layer_mapping.items():
            print(f"\nProcessing {service_layer_name}...")
            
            # Find the matching service layer
            service_layer = None
            for lyr in flc.layers:
                if lyr.properties.name == service_layer_name:
                    service_layer = lyr
                    break
            
            if service_layer is None:
                print(f" Warning: Could not find layer '{service_layer_name}' in service")
                continue
            
            # Get current count
            try:
                old_count = service_layer.query(return_count_only=True)
                print(f"  Current features: {old_count}")
            except:
                old_count = "unknown"
            
            # Step 1: Delete all features
            print(f"  Deleting all existing features...")
            try:
                service_layer.delete_features(where='1=1')
                print(f"  ✓ Deleted successfully")
            except Exception as e:
                print(f"  ✗ Deleted failed: {e}")
                continue
            
            # Step 2: APPEND (add new features from GDB)
            print(f"  Appending new features from GDB...")
            gdb_fc_path = os.path.join(gdb_path, gdb_layer_name)

            import arcpy
            arcpy.env.overwriteOutput = True

            if not arcpy.Exists(gdb_fc_path):
                print(f"  ✗ GDB feature class not found: {gdb_fc_path}")
                continue

            try:
                service_url = service_layer.url
                arcpy.management.Append(
                    inputs=gdb_fc_path,
                    target=service_url,
                    schema_type="NO_TEST"
                )
                
                # Verify new count
                new_count = service_layer.query(return_count_only=True)
                print(f"  ✓ Appended successfully: {new_count} features")
                
            except Exception as e:
                print(f"  ✗ Append failed: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        # Final verification
        print(f"\n{'='*70}")
        print("FINAL FEATURE COUNTS:")
        print(f"{'='*70}")
        for i, layer in enumerate(flc.layers):
            try:
                count = layer.query(return_count_only=True)
                print(f"  [{i}] {layer.properties.name}: {count} features")
            except:
                pass
        
        print(f"\n{'='*70}")
        print("✓ UPDATE COMPLETE")
        print(f"{'='*70}")
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return False


# Execute the update
import shutil

# Custom temp directory location
# Temp dir from config — not hardcoded
temp_dir = os.path.join(config.LOG_DIR, '..', 'Temp')
local_gdb_path = os.path.join(temp_dir, "iNat_Biosecurity_Plants_temp.gdb")

# Create temp directory if it doesn't exist
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)

print(f"\nCopying GDB to local temp directory...")
print(f"  From: {output_gdb}")
print(f"  To:   {local_gdb_path}")

try:
    # Remove old temp GDB if it exists
    if os.path.exists(local_gdb_path):
        try:
            shutil.rmtree(local_gdb_path)
        except:
            print("  (Overwriting existing temp GDB)")
    
    # Copy the GDB to local temp location
    shutil.copytree(output_gdb, local_gdb_path)
    print("✓ GDB copied successfully")
    
    # Update service
    success = update_hosted_service(SERVICE_ITEM_ID, local_gdb_path)
    
    if success:
        # Only update tracking file if AGOL update succeeded
        save_current_observations(tracking_file, current_obs_ids)
        print("\n✓ Observation tracking updated for next run")
    else:
        print("\nAGOL update failed - observation tracking NOT updated")
        print("  Next run will re-process these observations")
    
    # Clean up temp GDB
    print(f"\nCleaning up temp GDB...")
    try:
        time.sleep(2)
        shutil.rmtree(local_gdb_path)
        print("✓ Temp GDB removed")
    except:
        print("Temp GDB still in use - will be cleaned up on next run")
    
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

Connecting to ArcGIS Online...
✓ Connected as: CTregurtha_Nairn_HorizonsRC

Copying GDB to local temp directory...
  From: \\gisdata\GIS\Department\Environmental_Management\Biodiversity\BioData\Biosecurity\iNaturalist\iNat_Biosecurity_Plants.gdb
  To:   D:\Scripts\iNaturalist\Logs\..\Temp\iNat_Biosecurity_Plants_temp.gdb
✓ GDB copied successfully

UPDATING HOSTED FEATURE SERVICE

Found service: iNat_RPMP_Observations

Service has 4 layers

Processing iNat_AllProgrammes...
  Current features: 517
  Deleting all existing features...
  ✓ Deleted successfully
  Appending new features from GDB...
  ✓ Appended successfully: 517 features

Processing iNat_Eradication...
  Current features: 137
  Deleting all existing features...
  ✓ Deleted successfully
  Appending new features from GDB...
  ✓ Appended successfully: 137 features

Processing iNat_ProgressiveContainment...
  Current features: 377
  Deleting all existing features...
  ✓ Deleted successfully
  Appending new features from GDB...
  

Traceback (most recent call last):
﻿  File "C:/Users/CTREGU~1/AppData/Local/Temp/ArcGISProTemp27096/xpython_27096/422727953.py", line 159, in <module>
    save_current_observations(tracking_file, current_obs_ids)
    ^^^^^^^^^^^^^^^^^^^^^^^^^
﻿NameError: name 'save_current_observations' is not defined
﻿

## PREVIEW EMAIL TEMPLATES (Optional - for testing appearance)

In [23]:
# ============================================================================
# PREVIEW EMAIL TEMPLATES (Optional - for testing appearance)
# ============================================================================
# Note: staff names below are fictional placeholders for testing only.
# Real staff names and emails are stored in config.py.
# ============================================================================

from datetime import datetime
import os

# Sample notification data for preview — fictional placeholders
sample_eradication_obs = [
    {
        'observation_id': 123456789,
        'observation_url': 'https://www.inaturalist.org/observations/123456789',
        'programme': 'Eradication',
        'taxon_name': 'Gunnera tinctoria',
        'species_name': 'Chilean Rhubarb',
        'observed_on': '2026-01-06',
        'operating_area': 'Horowhenua',
        'staff_member': 'Jane Smith',
        'quality_grade': 'research',
        'flowers_fruits': 'Flowering',
        'description': 'Large patch near stream, approximately 20 plants'
    }
]

sample_exclusion_obs = [
    {
        'observation_id': 987654321,
        'observation_url': 'https://www.inaturalist.org/observations/987654321',
        'programme': 'Exclusion',
        'taxon_name': 'Sagittaria platyphylla',
        'species_name': 'Arrowhead',
        'observed_on': '2026-01-10',
        'operating_area': 'Palmerston North',
        'staff_member': 'John Doe',
        'quality_grade': 'research',
        'flowers_fruits': 'Flowering',
        'description': 'Found in wetland area'
    }
]

sample_weekly_obs = sample_eradication_obs + sample_exclusion_obs

def generate_preview_html(subject, html_body, filename):
    """Save email HTML to file for preview"""
    preview_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>{subject}</title>
        <style>
            body {{ margin: 0; padding: 20px; background-color: #e0e0e0; font-family: Arial, sans-serif; }}
            .email-preview {{ max-width: 800px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
            .preview-header {{ background: #333; color: white; padding: 15px; margin: -20px -20px 20px -20px; border-radius: 8px 8px 0 0; }}
            .preview-subject {{ font-size: 18px; font-weight: bold; margin-bottom: 5px; }}
            .preview-meta {{ font-size: 14px; color: #ccc; }}
        </style>
    </head>
    <body>
        <div class="email-preview">
            <div class="preview-header">
                <div class="preview-subject">Subject: {subject}</div>
                <div class="preview-meta">From: Horizons Biosecurity Team</div>
                <div class="preview-meta">To: Staff Member</div>
            </div>
            {html_body}
        </div>
    </body>
    </html>
    """

    preview_path = os.path.join(r"D:\Scripts\iNaturalist\Temp", filename)
    with open(preview_path, 'w', encoding='utf-8') as f:
        f.write(preview_html)
    return preview_path

import sys
sys.path.append(r'D:\Scripts\iNaturalist')
import inat_email_notifications

print("Generating email previews...")
print("="*70)

# 1. Eradication Alert
subject1, html1 = inat_email_notifications.create_immediate_alert_email(
    sample_eradication_obs, "Jane Smith", "Horowhenua"
)
preview1 = generate_preview_html(subject1, html1, "preview_eradication_alert.html")
print(f"\n1. Eradication Alert Email")
print(f"   {preview1}")

# 2. Exclusion Alert
subject2, html2 = inat_email_notifications.create_immediate_alert_email(
    sample_exclusion_obs, "John Doe", "Palmerston North"
)
preview2 = generate_preview_html(subject2, html2, "preview_exclusion_alert.html")
print(f"\n2. Exclusion Alert Email")
print(f"   {preview2}")

# 3. Weekly Summary
subject3, html3 = inat_email_notifications.create_weekly_summary_email(
    sample_weekly_obs, "Jane Smith", "Horowhenua"
)
preview3 = generate_preview_html(subject3, html3, "preview_weekly_summary.html")
print(f"\n3. Weekly Summary Email")
print(f"   {preview3}")

print("\n" + "="*70)
print("✓ Open these HTML files in your browser to preview the emails!")

Generating email previews...

1. Eradication Alert Email
   D:\Scripts\iNaturalist\Temp\preview_eradication_alert.html

2. Exclusion Alert Email
   D:\Scripts\iNaturalist\Temp\preview_exclusion_alert.html

3. Weekly Summary Email
   D:\Scripts\iNaturalist\Temp\preview_weekly_summary.html

✓ Open these HTML files in your browser to preview the emails!


## CREATE TEST NOTIFICATION FILE (for email testing)

In [25]:
# ============================================================================
# CREATE TEST NOTIFICATION FILE (for email testing)
# ============================================================================
# Note: staff names below are fictional placeholders for testing only.
# Real staff names and emails are stored in config.py (not on GitHub).
# ============================================================================

import json
from datetime import datetime

# Fictional placeholder test data
test_notifications = {
    'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_notifications': 2,
    'notifications': [
        {
            'observation_id': 123456789,
            'observation_url': 'https://www.inaturalist.org/observations/123456789',
            'programme': 'Eradication',
            'taxon_name': 'Gunnera tinctoria',
            'species_name': 'Chilean Rhubarb',
            'observed_on': '2024-01-06',
            'operating_area': 'Horowhenua',
            'staff_member': 'Jane Smith',
            'latitude': -40.5,
            'longitude': 175.5,
            'quality_grade': 'research',
            'user_name': 'Test Observer',
            'description': 'Test observation for email testing',
            'flowers_fruits': 'Flowering',
            'photoURL_1': '',
            'designation': 'AMZ'
        },
        {
            'observation_id': 987654321,
            'observation_url': 'https://www.inaturalist.org/observations/987654321',
            'programme': 'Exclusion',
            'taxon_name': 'Sagittaria platyphylla',
            'species_name': 'Arrowhead',
            'observed_on': '2024-01-10',
            'operating_area': 'Palmerston North',
            'staff_member': 'John Doe',
            'latitude': -40.3,
            'longitude': 175.6,
            'quality_grade': 'research',
            'user_name': 'Another Observer',
            'description': 'Found in wetland',
            'flowers_fruits': '',
            'photoURL_1': '',
            'designation': 'AMZ'
        }
    ]
}

# Notification file path from config — not hardcoded
notification_file = config.NOTIFICATION_FILE

with open(notification_file, 'w') as f:
    json.dump(test_notifications, f, indent=2)

print("✓ Test notification file created")
print(f"  Location: {notification_file}")
print(f"  Contains {len(test_notifications['notifications'])} test observations")

✓ Test notification file created
  Location: \\gisdata\GIS\Department\Environmental_Management\Biodiversity\BioData\Biosecurity\iNaturalist\pending_notifications.json
  Contains 2 test observations


## Send Email Notifications

In [26]:
# ============================================================================
# Send Email Notifications
# ============================================================================

import sys
sys.path.append(r'D:\Scripts\iNaturalist')
import inat_email_notifications  # ← Changed from email_notifications

# Path to notification file
notification_file = r"\\gisdata\GIS\Department\Environmental_Management\Biodiversity\BioData\Biosecurity\iNaturalist\pending_notifications.json"

# Process and send notifications
results = inat_email_notifications.process_notifications(notification_file)  

# Display results
if results['notifications_processed'] > 0:
    print(f"Processed {results['notifications_processed']} notifications")
    print(f"Sent {results['emails_sent']} emails successfully")
    if results['emails_failed'] > 0:
        print(f"Failed to send {results['emails_failed']} emails")

EMAIL NOTIFICATION SYSTEM

⚠️  TESTING MODE ENABLED
All emails will be sent to: courtney.tregurtha-nairn@horizons.govt.nz
Forcing notifications for observation IDs: [333948185, 74909956]

Filtered to 2 test observation(s)

📧 Processing 2 notification(s)...

Processing notifications for Abi Wightman...
  Using test email: courtney.tregurtha-nairn@horizons.govt.nz
  Sending Eradication alert (1 obs)...
  ✓ Sent to courtney.tregurtha-nairn@horizons.govt.nz

Processing notifications for Rory Johnson...
  Using test email: courtney.tregurtha-nairn@horizons.govt.nz
  Sending Exclusion alert (1 obs)...
  ✓ Sent to courtney.tregurtha-nairn@horizons.govt.nz

NOTIFICATION SUMMARY
Emails sent successfully: 2
Emails failed: 0

⚠ Testing mode - pending notifications NOT cleared


Processed 2 notifications
Sent 2 emails successfully
